# Lightweight KPR vs. BGE retrieval evaluation

Inference-only comparison of the actual `knowledgeable-ai/kpr-bge-base-en-v1.5` checkpoint and `BAAI/bge-base-en-v1.5` on a deterministic, labeled SciFact sample. No training or paid APIs.

**Kaggle:** enable Internet, optionally select T4/P100, then run all cells. GPU is optional. The report is `/kaggle/working/kpr_eval_report.json`.

The entity table is large. The notebook inspects Hub file sizes and free disk before downloading, uses `/kaggle/temp` caches, and stops at a 25-minute soft deadline to preserve time for the report. A 30-minute run is not guaranteed. If both labeled SciFact sources fail, it uses a 3-query `diagnostic_only` fixture and does not present those values as benchmark metrics.

In [1]:
from pathlib import Path
import base64, json, os, sys, tempfile, time

IS_KAGGLE = Path("/kaggle/working").is_dir()
RUNTIME_ROOT = Path("/kaggle/working/kpr_eval_project") if IS_KAGGLE else Path.cwd() / "kpr_eval_project"
OUTPUT_PATH = Path("/kaggle/working/kpr_eval_report.json") if IS_KAGGLE else Path.cwd() / "kpr_eval_report.json"
CACHE_ROOT = Path("/kaggle/temp/kpr_eval_cache") if Path("/kaggle/temp").is_dir() else Path(tempfile.gettempdir()) / "kpr_eval_cache"
HF_CACHE, DATA_CACHE = CACHE_ROOT / "hub", CACHE_ROOT / "dataset"
for directory in (RUNTIME_ROOT / "src", CACHE_ROOT, HF_CACHE, DATA_CACHE, CACHE_ROOT / "modules", CACHE_ROOT / "tmp", OUTPUT_PATH.parent):
    directory.mkdir(parents=True, exist_ok=True)
os.environ.update({"HF_HOME": str(CACHE_ROOT), "HF_HUB_CACHE": str(HF_CACHE), "HF_MODULES_CACHE": str(CACHE_ROOT / "modules"), "TRANSFORMERS_CACHE": str(HF_CACHE), "TMPDIR": str(CACHE_ROOT / "tmp"), "HF_HUB_DISABLE_TELEMETRY": "1"})
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "45")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")

# Embedded helpers keep this notebook runnable when uploaded without the src/ folder.
EMBEDDED_SOURCE_B64 = {"src/data.py": "...", "src/models.py": "IiIiSHVnZ2luZyBGYWNlIGNoZWNrcG9pbnQgaW5zcGVjdGlvbiwgYm91bmRlZCBkb3dubG9hZCwgYW5kIGVtYmVkZGluZyBoZWxwZXJzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKaW1wb3J0IHV1aWQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBudW1weSBhcyBucAoKCktQUl9NT0RFTF9JRCA9ICJrbm93bGVkZ2VhYmxlLWFpL2twci1iZ2UtYmFzZS1lbi12MS41IgpCQVNFTElORV9NT0RFTF9JRCA9ICJCQUFJL2JnZS1iYXNlLWVuLXYxLjUiCktQUl9TT1VSQ0VfRklMRVMgPSAoImNvbmZpZ3VyYXRpb24ucHkiLCAibW9kZWxpbmcucHkiLCAidG9rZW5pemF0aW9uX2twci5weSIpCgoKZGVmIGluc3BlY3RfaHViX21vZGVsKG1vZGVsX2lkOiBzdHIpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgIiIiUmVhZCByZXBvc2l0b3J5IHJldmlzaW9uIGFuZCBmaWxlIHNpemVzIGZyb20gdGhlIEh1Z2dpbmcgRmFjZSBIdWIgQVBJLiIiIgogICAgZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IEhmQXBpCgogICAgaW5mbyA9IEhmQXBpKCkubW9kZWxfaW5mbyhtb2RlbF9pZCwgZmlsZXNfbWV0YWRhdGE9VHJ1ZSkKICAgIGZpbGVzID0gW10KICAgIHVua25vd25fZmlsZXMgPSBbXQogICAgZm9yIHNpYmxpbmcgaW4gaW5mby5zaWJsaW5nczoKICAgICAgICBzaXplID0gZ2V0YXR0cihzaWJsaW5nLCAic2l6ZSIsIE5vbmUpCiAgICAgICAgaWYgc2l6ZSBpcyBOb25lOgogICAgICAgICAgICBsZnMgPSBnZXRhdHRyKHNpYmxpbmcsICJsZnMiLCBOb25lKQogICAgICAgICAgICBzaXplID0gZ2V0YXR0cihsZnMsICJzaXplIiwgTm9uZSkgaWYgbGZzIGlzIG5vdCBOb25lIGVsc2UgTm9uZQogICAgICAgIGlmIHNpemUgaXMgTm9uZToKICAgICAgICAgICAgdW5rbm93bl9maWxlcy5hcHBlbmQoc2libGluZy5yZmlsZW5hbWUpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZmlsZXMuYXBwZW5kKHsibmFtZSI6IHNpYmxpbmcucmZpbGVuYW1lLCAic2l6ZV9ieXRlcyI6IGludChzaXplKX0pCiAgICByZXR1cm4gewogICAgICAgICJtb2RlbF9pZCI6IG1vZGVsX2lkLAogICAgICAgICJyZXZpc2lvbiI6IGluZm8uc2hhLAogICAgICAgICJ0b3RhbF9yZXBvc2l0b3J5X3NpemVfYnl0ZXMiOiBzdW0oaXRlbVsic2l6ZV9ieXRlcyJdIGZvciBpdGVtIGluIGZpbGVzKSwKICAgICAgICAiZmlsZXMiOiBmaWxlcywKICAgICAgICAidW5rbm93bl9zaXplX2ZpbGVzIjogdW5rbm93bl9maWxlcywKICAgICAgICAiY29tbWl0X3VybCI6IGYiaHR0cHM6Ly9odWdnaW5nZmFjZS5jby97bW9kZWxfaWR9L3RyZWUve2luZm8uc2hhfSIsCiAgICB9CgoKZGVmIGluc3BlY3Rfa3ByX3JlbW90ZV9jb2RlKG1vZGVsX2lkOiBzdHIsIHJldmlzaW9uOiBzdHIsIGNhY2hlX2Rpcjogc3RyIHwgUGF0aCkgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJEb3dubG9hZCBvbmx5IHNtYWxsIGN1c3RvbS1jb2RlIGZpbGVzIGFuZCBjaGVjayBleHBlY3RlZCBpbmZlcmVuY2UgaG9va3MuCgogICAgVGhlc2UgY2hlY2tzIGhhcHBlbiBiZWZvcmUgdGhlIG5vdGVib29rIGxvYWRzIHRoZSByZW1vdGUgaW1wbGVtZW50YXRpb24gd2l0aAogICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZS4gU291cmNlIGlzIHBpbm5lZCB0byB0aGUgaW5zcGVjdGVkIEh1YiByZXZpc2lvbi4KICAgICIiIgogICAgZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IGhmX2h1Yl9kb3dubG9hZAoKICAgIGNhY2hlX2RpciA9IFBhdGgoY2FjaGVfZGlyKQogICAgc291cmNlID0ge30KICAgIHNvdXJjZV9oYXNoZXMgPSB7fQogICAgZm9yIGZpbGVuYW1lIGluIEtQUl9TT1VSQ0VfRklMRVM6CiAgICAgICAgbG9jYWxfcGF0aCA9IFBhdGgoCiAgICAgICAgICAgIGhmX2h1Yl9kb3dubG9hZCgKICAgICAgICAgICAgICAgIHJlcG9faWQ9bW9kZWxfaWQsCiAgICAgICAgICAgICAgICBmaWxlbmFtZT1maWxlbmFtZSwKICAgICAgICAgICAgICAgIHJldmlzaW9uPXJldmlzaW9uLAogICAgICAgICAgICAgICAgY2FjaGVfZGlyPXN0cihjYWNoZV9kaXIpLAogICAgICAgICAgICAgICAgZXRhZ190aW1lb3V0PTMwLAogICAgICAgICAgICApCiAgICAgICAgKQogICAgICAgIGNvbnRlbnRzID0gbG9jYWxfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICBzb3VyY2VbZmlsZW5hbWVdID0gY29udGVudHMKICAgICAgICBzb3VyY2VfaGFzaGVzW2ZpbGVuYW1lXSA9IF9zaGEyNTZfdGV4dChjb250ZW50cykKCiAgICBtb2RlbF9jb2RlID0gc291cmNlWyJtb2RlbGluZy5weSJdCiAgICB0b2tlbml6ZXJfY29kZSA9IHNvdXJjZVsidG9rZW5pemF0aW9uX2twci5weSJdCiAgICBjb25maWdfY29kZSA9IHNvdXJjZVsiY29uZmlndXJhdGlvbi5weSJdCiAgICBjaGVja3MgPSB7CiAgICAgICAgImVudGl0eV9mdXNpb25fYXR0ZW50aW9uX3ByZXNlbnQiOiAiRW50aXR5RnVzaW9uTXVsdGlIZWFkQXR0ZW50aW9uIiBpbiBtb2RlbF9jb2RlLAogICAgICAgICJlbnRpdHlfZnVzaW9uX2xheWVyX3ByZXNlbnQiOiAiRW50aXR5RnVzaW9uTGF5ZXIiIGluIG1vZGVsX2NvZGUsCiAgICAgICAgImVudGl0eV9pbnB1dHNfY29uc3VtZWQiOiAiZW50aXR5X2lkcyIgaW4gbW9kZWxfY29kZSBhbmQgImVudGl0eV9lbWJlZHMiIGluIG1vZGVsX2NvZGUsCiAgICAgICAgIm1lbW9yeV9tYXBwZWRfZW50aXR5X2VtYmVkZGluZ3MiOiAnbW1hcF9tb2RlPSJyIicgaW4gdG9rZW5pemVyX2NvZGUgb3IgIm1tYXBfbW9kZT0nciciIGluIHRva2VuaXplcl9jb2RlLAogICAgICAgICJjdXN0b21fYmVydF9jb25maWd1cmF0aW9uX3ByZXNlbnQiOiAiS1BSQ29uZmlnRm9yQmVydCIgaW4gY29uZmlnX2NvZGUsCiAgICAgICAgImVudGl0eV9saW5raW5nX2FuZF9wb3NpdGlvbnNfcHJlc2VudCI6ICgKICAgICAgICAgICAgImRldGVjdF9tZW50aW9ucyIgaW4gdG9rZW5pemVyX2NvZGUgYW5kICJlbnRpdHlfcG9zaXRpb25faWRzIiBpbiB0b2tlbml6ZXJfY29kZQogICAgICAgICksCiAgICAgICAgIm1hcmlzYV90cmllX2RlcGVuZGVuY3lfcHJlc2VudCI6ICJtYXJpc2FfdHJpZSIgaW4gdG9rZW5pemVyX2NvZGUsCiAgICAgICAgInNwYWN5X2RlcGVuZGVuY3lfcHJlc2VudCI6ICJpbXBvcnQgc3BhY3kiIGluIHRva2VuaXplcl9jb2RlLAogICAgfQogICAgbWlzc2luZyA9IFtuYW1lIGZvciBuYW1lLCBwYXNzZWQgaW4gY2hlY2tzLml0ZW1zKCkgaWYgbm90IHBhc3NlZF0KICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiS1BSIGN1c3RvbS1jb2RlIGluc3BlY3Rpb24gZGlkIG5vdCBwYXNzIGV4cGVjdGVkIGNoZWNrczoge21pc3Npbmd9IikKCiAgICByZXR1cm4gewogICAgICAgICJyZXZpc2lvbiI6IHJldmlzaW9uLAogICAgICAgICJzb3VyY2VfZmlsZXMiOiBsaXN0KEtQUl9TT1VSQ0VfRklMRVMpLAogICAgICAgICJzb3VyY2Vfc2hhMjU2Ijogc291cmNlX2hhc2hlcywKICAgICAgICAiY2hlY2tzIjogY2hlY2tzLAogICAgICAgICJpbmZlcmVuY2Vfb2JzZXJ2YXRpb24iOiAoCiAgICAgICAgICAgICJUaGUgSHViIHRva2VuaXplciBsaW5rcyBkZXRlY3RlZCBlbnRpdHkgbWVudGlvbnMgdG8gZW50aXR5IElEcyBhbmQgdG9rZW4gcG9zaXRpb25zLCAiCiAgICAgICAgICAgICJyZWFkcyB0aGUgbGFyZ2UgZW50aXR5IHRhYmxlIHdpdGggTnVtUHkgbWVtb3J5IG1hcHBpbmcsIGFuZCBiYXRjaGVzIHNlbGVjdGVkIGVudGl0eSAiCiAgICAgICAgICAgICJ2ZWN0b3JzIGludG8gZW50aXR5X2VtYmVkcy4gVGhlIEtQUiBtb2RlbCBlbmNvZGVzIENMUyB0ZXh0IHZlY3RvcnMgYW5kIGZ1c2VzIGVudGl0eSAiCiAgICAgICAgICAgICJ2ZWN0b3JzIHdpdGggY29udGV4dCB1c2luZyBlbnRpdHktZnVzaW9uIGF0dGVudGlvbi4iCiAgICAgICAgKSwKICAgICAgICAic291cmNlX2NvZGVfZXhlY3V0ZWRfZHVyaW5nX2luc3BlY3Rpb24iOiBGYWxzZSwKICAgIH0KCgpkZWYgcmVxdWlyZWRfc25hcHNob3RfZmlsZXMobW9kZWxfaW5mbzogZGljdFtzdHIsIEFueV0pIC0+IHR1cGxlW2xpc3Rbc3RyXSwgaW50XToKICAgICIiIlJldHVybiBpbmZlcmVuY2UgZmlsZXMgYW5kIHRoZWlyIGtub3duIGJ5dGUgdG90YWwsIGV4Y2x1ZGluZyBjYXJkIG1ldGFkYXRhLiIiIgogICAgZXhjbHVkZWQgPSB7IlJFQURNRS5tZCIsICIuZ2l0YXR0cmlidXRlcyJ9CiAgICByZXF1aXJlZCA9IFtpdGVtIGZvciBpdGVtIGluIG1vZGVsX2luZm9bImZpbGVzIl0gaWYgaXRlbVsibmFtZSJdIG5vdCBpbiBleGNsdWRlZF0KICAgIHVua25vd25fcmVxdWlyZWQgPSBbbmFtZSBmb3IgbmFtZSBpbiBtb2RlbF9pbmZvWyJ1bmtub3duX3NpemVfZmlsZXMiXSBpZiBuYW1lIG5vdCBpbiBleGNsdWRlZF0KICAgIGlmIHVua25vd25fcmVxdWlyZWQ6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiQ2Fubm90IGVzdGltYXRlIHJlcXVpcmVkIGNoZWNrcG9pbnQgZGlzayB1c2FnZTsgbWlzc2luZyBzaXplcyBmb3Ige3Vua25vd25fcmVxdWlyZWR9LiIpCiAgICBpZiBub3QgcmVxdWlyZWQ6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiSHViIEFQSSByZXR1cm5lZCBubyBzaXplZCBpbmZlcmVuY2UgZmlsZXMgZm9yIHttb2RlbF9pbmZvWydtb2RlbF9pZCddfS4iKQogICAgcmV0dXJuIFtpdGVtWyJuYW1lIl0gZm9yIGl0ZW0gaW4gcmVxdWlyZWRdLCBzdW0oaXRlbVsic2l6ZV9ieXRlcyJdIGZvciBpdGVtIGluIHJlcXVpcmVkKQoKCmRlZiBjYWNoZV9zaXplX2J5dGVzKHBhdGg6IHN0ciB8IFBhdGgpIC0+IGludDoKICAgICIiIlN1bSByZWd1bGFyLWZpbGUgc2l6ZXMgd2l0aG91dCBjb3VudGluZyBzeW1saW5rIHRhcmdldHMgdHdpY2UuIiIiCiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgaWYgbm90IHBhdGguZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIDAKICAgIHRvdGFsID0gMAogICAgc2Vlbjogc2V0W3N0cl0gPSBzZXQoKQogICAgZm9yIGZpbGVfcGF0aCBpbiBwYXRoLnJnbG9iKCIqIik6CiAgICAgICAgaWYgbm90IGZpbGVfcGF0aC5pc19maWxlKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXNvbHZlZCA9IHN0cihmaWxlX3BhdGgucmVzb2x2ZSgpKQogICAgICAgICAgICBpZiByZXNvbHZlZCBpbiBzZWVuOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgc2Vlbi5hZGQocmVzb2x2ZWQpCiAgICAgICAgICAgIHRvdGFsICs9IGZpbGVfcGF0aC5zdGF0KCkuc3Rfc2l6ZQogICAgICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgICAgICBjb250aW51ZQogICAgcmV0dXJuIHRvdGFsCgoKZGVmIGRvd25sb2FkX3NuYXBzaG90X3dpdGhfZGVhZGxpbmUoCiAgICBtb2RlbF9pZDogc3RyLAogICAgcmV2aXNpb246IHN0ciwKICAgIGNhY2hlX2Rpcjogc3RyIHwgUGF0aCwKICAgIGFsbG93X3BhdHRlcm5zOiBsaXN0W3N0cl0sCiAgICAqLAogICAgc3RvcF9hdF9tb25vdG9uaWM6IGZsb2F0LAogICAgcG9sbF9zZWNvbmRzOiBmbG9hdCA9IDIuMCwKICAgIG1heF93b3JrZXJzOiBpbnQgPSA0LAopIC0+IHN0cjoKICAgICIiIkRvd25sb2FkIHRocm91Z2ggYSBjaGlsZCBwcm9jZXNzIHNvIGEgc2xvdyBIdWIgcmVxdWVzdCBjYW4gYmUgc3RvcHBlZC4KCiAgICBUaGUgcGFyZW50IG5vdGVib29rIHJlbWFpbnMgYWJsZSB0byBzYXZlIGl0cyBwYXJ0aWFsIEpTT04gcmVwb3J0IGlmIHRoZSBzb2Z0CiAgICBkZWFkbGluZSBpcyByZWFjaGVkLiBBIHByb2Nlc3MgZXhpdCBjYXVzZWQgYnkgbmV0d29yay9kaXNrIGZhaWx1cmUgaXMgcmFpc2VkCiAgICBhcyBhbiBlcnJvciBhbmQgbmV2ZXIgdHJlYXRlZCBhcyBhIHN1Y2Nlc3NmdWwgS1BSIGxvYWQuCiAgICAiIiIKICAgIGNhY2hlX2RpciA9IFBhdGgoY2FjaGVfZGlyKQogICAgY2FjaGVfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG1hcmtlcl9wYXRoID0gY2FjaGVfZGlyIC8gZiJzbmFwc2hvdC1wYXRoLXt1dWlkLnV1aWQ0KCkuaGV4fS50eHQiCiAgICBkb3dubG9hZF9zY3JpcHQgPSAoCiAgICAgICAgImltcG9ydCBqc29uLCBzeXNcbiIKICAgICAgICAiZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IHNuYXBzaG90X2Rvd25sb2FkXG4iCiAgICAgICAgInJlcG9faWQsIHJldmlzaW9uLCBjYWNoZV9kaXIsIHBhdHRlcm5zX2pzb24sIG1hcmtlcl9wYXRoLCB3b3JrZXJzID0gc3lzLmFyZ3ZbMTpdXG4iCiAgICAgICAgInBhdGggPSBzbmFwc2hvdF9kb3dubG9hZChyZXBvX2lkPXJlcG9faWQsIHJldmlzaW9uPXJldmlzaW9uLCBjYWNoZV9kaXI9Y2FjaGVfZGlyLCAiCiAgICAgICAgImFsbG93X3BhdHRlcm5zPWpzb24ubG9hZHMocGF0dGVybnNfanNvbiksIG1heF93b3JrZXJzPWludCh3b3JrZXJzKSwgZXRhZ190aW1lb3V0PTMwKVxuIgogICAgICAgICJvcGVuKG1hcmtlcl9wYXRoLCAndycsIGVuY29kaW5nPSd1dGYtOCcpLndyaXRlKHBhdGgpXG4iCiAgICApCiAgICBwcm9jZXNzX2VudiA9IG9zLmVudmlyb24uY29weSgpCiAgICBwcm9jZXNzX2Vudi5zZXRkZWZhdWx0KCJIRl9IVUJfRE9XTkxPQURfVElNRU9VVCIsICI0NSIpCiAgICBwcm9jZXNzX2Vudi5zZXRkZWZhdWx0KCJIRl9IVUJfRVRBR19USU1FT1VUIiwgIjMwIikKICAgIGNvbW1hbmQgPSBbCiAgICAgICAgc3lzLmV4ZWN1dGFibGUsCiAgICAgICAgIi11IiwKICAgICAgICAiLWMiLAogICAgICAgIGRvd25sb2FkX3NjcmlwdCwKICAgICAgICBtb2RlbF9pZCwKICAgICAgICByZXZpc2lvbiwKICAgICAgICBzdHIoY2FjaGVfZGlyKSwKICAgICAgICBqc29uLmR1bXBzKGFsbG93X3BhdHRlcm5zKSwKICAgICAgICBzdHIobWFya2VyX3BhdGgpLAogICAgICAgIHN0cihtYXhfd29ya2VycyksCiAgICBdCiAgICBwcm9jZXNzID0gc3VicHJvY2Vzcy5Qb3Blbihjb21tYW5kLCBlbnY9cHJvY2Vzc19lbnYpCiAgICB0cnk6CiAgICAgICAgd2hpbGUgcHJvY2Vzcy5wb2xsKCkgaXMgTm9uZToKICAgICAgICAgICAgaWYgdGltZS5tb25vdG9uaWMoKSA+PSBzdG9wX2F0X21vbm90b25pYzoKICAgICAgICAgICAgICAgIHByb2Nlc3MudGVybWluYXRlKCkKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwcm9jZXNzLndhaXQodGltZW91dD0xMCkKICAgICAgICAgICAgICAgIGV4Y2VwdCBzdWJwcm9jZXNzLlRpbWVvdXRFeHBpcmVkOgogICAgICAgICAgICAgICAgICAgIHByb2Nlc3Mua2lsbCgpCiAgICAgICAgICAgICAgICAgICAgcHJvY2Vzcy53YWl0KHRpbWVvdXQ9NSkKICAgICAgICAgICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcigKICAgICAgICAgICAgICAgICAgICBmIkRvd25sb2FkIG9mIHttb2RlbF9pZH0gZGlkIG5vdCBmaW5pc2ggYmVmb3JlIHRoZSBub3RlYm9vayBzb2Z0IGRlYWRsaW5lOyAiCiAgICAgICAgICAgICAgICAgICAgInBhcnRpYWwgSHVnZ2luZyBGYWNlIGNhY2hlIGZpbGVzIG1heSByZW1haW4gaW4gdGhlIGRpc3Bvc2FibGUgY2FjaGUuIgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICB0aW1lLnNsZWVwKHBvbGxfc2Vjb25kcykKICAgICAgICBpZiBwcm9jZXNzLnJldHVybmNvZGUgIT0gMDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiSHVnZ2luZyBGYWNlIHNuYXBzaG90IGRvd25sb2FkIGZhaWxlZCB3aXRoIGNoaWxkIGV4aXQgY29kZSB7cHJvY2Vzcy5yZXR1cm5jb2RlfS4iKQogICAgICAgIGlmIG5vdCBtYXJrZXJfcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJIdWdnaW5nIEZhY2UgZG93bmxvYWRlciBleGl0ZWQgd2l0aG91dCByZXR1cm5pbmcgYSBzbmFwc2hvdCBwYXRoLiIpCiAgICAgICAgc25hcHNob3RfcGF0aCA9IG1hcmtlcl9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKS5zdHJpcCgpCiAgICAgICAgaWYgbm90IHNuYXBzaG90X3BhdGggb3Igbm90IFBhdGgoc25hcHNob3RfcGF0aCkuaXNfZGlyKCk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkh1Z2dpbmcgRmFjZSByZXR1cm5lZCBhbiBpbnZhbGlkIHNuYXBzaG90IHBhdGg6IHtzbmFwc2hvdF9wYXRoIXJ9IikKICAgICAgICByZXR1cm4gc25hcHNob3RfcGF0aAogICAgZmluYWxseToKICAgICAgICBtYXJrZXJfcGF0aC51bmxpbmsobWlzc2luZ19vaz1UcnVlKQoKCmRlZiBsb2FkX3NlbnRlbmNlX3RyYW5zZm9ybWVyKAogICAgbW9kZWxfcGF0aDogc3RyIHwgUGF0aCwKICAgICosCiAgICBkZXZpY2U6IHN0ciwKICAgIHRydXN0X3JlbW90ZV9jb2RlOiBib29sLAogICAgbWF4X3NlcV9sZW5ndGg6IGludCA9IDI1NiwKKSAtPiB0dXBsZVtBbnksIGRpY3Rbc3RyLCBBbnldXToKICAgICIiIkxvYWQgYSBsb2NhbCBTZW50ZW5jZVRyYW5zZm9ybWVyIGNoZWNrcG9pbnQgYW5kIHJlY29yZCBpdHMgbG9hZCBtZXRhZGF0YS4iIiIKICAgIGZyb20gc2VudGVuY2VfdHJhbnNmb3JtZXJzIGltcG9ydCBTZW50ZW5jZVRyYW5zZm9ybWVyCgogICAgc3RhcnRlZCA9IHRpbWUubW9ub3RvbmljKCkKICAgIGxvYWRfZmFsbGJhY2sgPSBOb25lCiAgICB0cnk6CiAgICAgICAgbW9kZWwgPSBTZW50ZW5jZVRyYW5zZm9ybWVyKAogICAgICAgICAgICBzdHIobW9kZWxfcGF0aCksCiAgICAgICAgICAgIHRydXN0X3JlbW90ZV9jb2RlPXRydXN0X3JlbW90ZV9jb2RlLAogICAgICAgICAgICBkZXZpY2U9ZGV2aWNlLAogICAgICAgICAgICBsb2NhbF9maWxlc19vbmx5PVRydWUsCiAgICAgICAgKQogICAgZXhjZXB0IFJ1bnRpbWVFcnJvciBhcyBleGM6CiAgICAgICAgaWYgIm91dCBvZiBtZW1vcnkiIG5vdCBpbiBzdHIoZXhjKS5sb3dlcigpIG9yIG5vdCBzdHIoZGV2aWNlKS5zdGFydHN3aXRoKCJjdWRhIik6CiAgICAgICAgICAgIHJhaXNlCiAgICAgICAgdHJ5OgogICAgICAgICAgICBpbXBvcnQgdG9yY2gKCiAgICAgICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwYXNzCiAgICAgICAgbG9hZF9mYWxsYmFjayA9ICJDVURBIG91dCBvZiBtZW1vcnkgZHVyaW5nIG1vZGVsIGxvYWRpbmc7IHJlbG9hZGVkIHRoZSBzYW1lIGNoZWNrcG9pbnQgb24gQ1BVLiIKICAgICAgICBtb2RlbCA9IFNlbnRlbmNlVHJhbnNmb3JtZXIoCiAgICAgICAgICAgIHN0cihtb2RlbF9wYXRoKSwKICAgICAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9dHJ1c3RfcmVtb3RlX2NvZGUsCiAgICAgICAgICAgIGRldmljZT0iY3B1IiwKICAgICAgICAgICAgbG9jYWxfZmlsZXNfb25seT1UcnVlLAogICAgICAgICkKICAgIG1vZGVsLm1heF9zZXFfbGVuZ3RoID0gbWF4X3NlcV9sZW5ndGgKICAgIGVsYXBzZWQgPSB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZAogICAgcGFyYW1ldGVycyA9IGxpc3QobW9kZWwucGFyYW1ldGVycygpKQogICAgZHR5cGUgPSBzdHIocGFyYW1ldGVyc1swXS5kdHlwZSkucmVwbGFjZSgidG9yY2guIiwgIiIpIGlmIHBhcmFtZXRlcnMgZWxzZSAidW5rbm93biIKICAgIHJldHVybiBtb2RlbCwgewogICAgICAgICJsb2FkX3NlY29uZHMiOiBlbGFwc2VkLAogICAgICAgICJlbWJlZGRpbmdfZGltZW5zaW9uIjogbW9kZWwuZ2V0X3NlbnRlbmNlX2VtYmVkZGluZ19kaW1lbnNpb24oKSwKICAgICAgICAiZGV2aWNlIjogc3RyKG1vZGVsLmRldmljZSksCiAgICAgICAgInJlcXVlc3RlZF9kZXZpY2UiOiBkZXZpY2UsCiAgICAgICAgImxvYWRfZmFsbGJhY2siOiBsb2FkX2ZhbGxiYWNrLAogICAgICAgICJwcmVjaXNpb24iOiBkdHlwZSwKICAgICAgICAibWF4X3NlcXVlbmNlX2xlbmd0aCI6IG1heF9zZXFfbGVuZ3RoLAogICAgfQoKCmRlZiBlbmNvZGVfdGV4dHNfYmF0Y2hlZCgKICAgIG1vZGVsOiBBbnksCiAgICB0ZXh0czogbGlzdFtzdHJdLAogICAgKiwKICAgIGJhdGNoX3NpemU6IGludCwKICAgIGRlYWRsaW5lX21vbm90b25pYzogZmxvYXQsCiAgICBkZXNjcmlwdGlvbjogc3RyLAopIC0+IHR1cGxlW25wLm5kYXJyYXksIGRpY3Rbc3RyLCBBbnldXToKICAgICIiIkVuY29kZSBpbiBib3VuZGVkIG1pY3JvYmF0Y2hlcywgcmVkdWNpbmcgdGhlIGJhdGNoIGFmdGVyIENVREEgT09NLiIiIgogICAgaWYgbm90IHRleHRzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJDYW5ub3QgZW5jb2RlIGFuIGVtcHR5IHtkZXNjcmlwdGlvbn0gY29sbGVjdGlvbi4iKQogICAgb3V0cHV0cyA9IFtdCiAgICBwb3NpdGlvbiA9IDAKICAgIGVmZmVjdGl2ZV9iYXRjaF9zaXplID0gbWF4KDEsIGludChiYXRjaF9zaXplKSkKICAgIG9vbV9yZXRyaWVzID0gMAogICAgY3B1X2ZhbGxiYWNrcyA9IDAKICAgIHN0YXJ0ZWQgPSB0aW1lLm1vbm90b25pYygpCiAgICB3aGlsZSBwb3NpdGlvbiA8IGxlbih0ZXh0cyk6CiAgICAgICAgaWYgdGltZS5tb25vdG9uaWMoKSA+PSBkZWFkbGluZV9tb25vdG9uaWM6CiAgICAgICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcihmIlNvZnQgZXhlY3V0aW9uIGRlYWRsaW5lIHJlYWNoZWQgd2hpbGUgZW5jb2Rpbmcge2Rlc2NyaXB0aW9ufS4iKQogICAgICAgIGNodW5rID0gdGV4dHNbcG9zaXRpb24gOiBwb3NpdGlvbiArIGVmZmVjdGl2ZV9iYXRjaF9zaXplXQogICAgICAgIHRyeToKICAgICAgICAgICAgZW1iZWRkaW5ncyA9IG1vZGVsLmVuY29kZSgKICAgICAgICAgICAgICAgIGNodW5rLAogICAgICAgICAgICAgICAgYmF0Y2hfc2l6ZT1sZW4oY2h1bmspLAogICAgICAgICAgICAgICAgY29udmVydF90b19udW1weT1UcnVlLAogICAgICAgICAgICAgICAgbm9ybWFsaXplX2VtYmVkZGluZ3M9VHJ1ZSwKICAgICAgICAgICAgICAgIHNob3dfcHJvZ3Jlc3NfYmFyPUZhbHNlLAogICAgICAgICAgICApCiAgICAgICAgZXhjZXB0IFJ1bnRpbWVFcnJvciBhcyBleGM6CiAgICAgICAgICAgIGlmICJvdXQgb2YgbWVtb3J5IiBub3QgaW4gc3RyKGV4YykubG93ZXIoKToKICAgICAgICAgICAgICAgIHJhaXNlCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGltcG9ydCB0b3JjaAoKICAgICAgICAgICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgYW5kIGVmZmVjdGl2ZV9iYXRjaF9zaXplID4gMToKICAgICAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKICAgICAgICAgICAgICAgICAgICBvb21fcmV0cmllcyArPSAxCiAgICAgICAgICAgICAgICAgICAgZWZmZWN0aXZlX2JhdGNoX3NpemUgPSBtYXgoMSwgZWZmZWN0aXZlX2JhdGNoX3NpemUgLy8gMikKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgcGFzcwogICAgICAgICAgICBpZiBzdHIoZ2V0YXR0cihtb2RlbCwgImRldmljZSIsICIiKSkuc3RhcnRzd2l0aCgiY3VkYSIpOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIG1vZGVsLnRvKCJjcHUiKQogICAgICAgICAgICAgICAgICAgIGltcG9ydCB0b3JjaAoKICAgICAgICAgICAgICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKICAgICAgICAgICAgICAgICAgICBjcHVfZmFsbGJhY2tzICs9IDEKICAgICAgICAgICAgICAgICAgICBvb21fcmV0cmllcyArPSAxCiAgICAgICAgICAgICAgICAgICAgZWZmZWN0aXZlX2JhdGNoX3NpemUgPSAxCiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZmFsbGJhY2tfZXJyb3I6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAgICAgICAgICAgICBmIntkZXNjcmlwdGlvbn0gcmFuIG91dCBvZiBHUFUgbWVtb3J5IGV2ZW4gYXQgYmF0Y2ggc2l6ZSAxLCBhbmQgQ1BVIGZhbGxiYWNrIGZhaWxlZDogIgogICAgICAgICAgICAgICAgICAgICAgICBmInt0eXBlKGZhbGxiYWNrX2Vycm9yKS5fX25hbWVfX306IHtmYWxsYmFja19lcnJvcn0iCiAgICAgICAgICAgICAgICAgICAgKSBmcm9tIGV4YwogICAgICAgICAgICByYWlzZQogICAgICAgIGVtYmVkZGluZ3MgPSBucC5hc2FycmF5KGVtYmVkZGluZ3MsIGR0eXBlPW5wLmZsb2F0MzIpCiAgICAgICAgaWYgZW1iZWRkaW5ncy5uZGltICE9IDIgb3IgZW1iZWRkaW5ncy5zaGFwZVswXSAhPSBsZW4oY2h1bmspOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJ7ZGVzY3JpcHRpb259IGVtYmVkZGluZ3MgaGF2ZSBpbnZhbGlkIHNoYXBlIHtlbWJlZGRpbmdzLnNoYXBlfTsgZXhwZWN0ZWQgKHtsZW4oY2h1bmspfSwgZGltKS4iCiAgICAgICAgICAgICkKICAgICAgICBpZiBub3QgbnAuaXNmaW5pdGUoZW1iZWRkaW5ncykuYWxsKCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7ZGVzY3JpcHRpb259IGVtYmVkZGluZ3MgY29udGFpbiBOYU4gb3IgaW5maW5pdGUgdmFsdWVzLiIpCiAgICAgICAgbm9ybXMgPSBucC5saW5hbGcubm9ybShlbWJlZGRpbmdzLCBheGlzPTEpCiAgICAgICAgaWYgbm90IG5wLmFsbGNsb3NlKG5vcm1zLCAxLjAsIGF0b2w9MmUtMyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7ZGVzY3JpcHRpb259IGVtYmVkZGluZ3MgYXJlIG5vdCBMMi1ub3JtYWxpemVkIGFzIHJlcXVlc3RlZC4iKQogICAgICAgIG91dHB1dHMuYXBwZW5kKGVtYmVkZGluZ3MpCiAgICAgICAgcG9zaXRpb24gKz0gbGVuKGNodW5rKQoKICAgIHJlc3VsdCA9IG5wLnZzdGFjayhvdXRwdXRzKQogICAgZWxhcHNlZCA9IHRpbWUubW9ub3RvbmljKCkgLSBzdGFydGVkCiAgICByZXR1cm4gcmVzdWx0LCB7CiAgICAgICAgIml0ZW1fY291bnQiOiBsZW4odGV4dHMpLAogICAgICAgICJzZWNvbmRzIjogZWxhcHNlZCwKICAgICAgICAic2Vjb25kc19wZXJfaXRlbSI6IGVsYXBzZWQgLyBsZW4odGV4dHMpLAogICAgICAgICJyZXF1ZXN0ZWRfYmF0Y2hfc2l6ZSI6IGludChiYXRjaF9zaXplKSwKICAgICAgICAiZWZmZWN0aXZlX2JhdGNoX3NpemUiOiBlZmZlY3RpdmVfYmF0Y2hfc2l6ZSwKICAgICAgICAib29tX3JldHJpZXMiOiBvb21fcmV0cmllcywKICAgICAgICAiY3B1X2ZhbGxiYWNrcyI6IGNwdV9mYWxsYmFja3MsCiAgICAgICAgImVmZmVjdGl2ZV9kZXZpY2UiOiBzdHIoZ2V0YXR0cihtb2RlbCwgImRldmljZSIsICJ1bmtub3duIikpLAogICAgICAgICJlbWJlZGRpbmdfZGltZW5zaW9uIjogaW50KHJlc3VsdC5zaGFwZVsxXSksCiAgICB9CgoKZGVmIF9zaGEyNTZfdGV4dCh2YWx1ZTogc3RyKSAtPiBzdHI6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYodmFsdWUuZW5jb2RlKCJ1dGYtOCIpKS5oZXhkaWdlc3QoKQo=", "src/evaluation.py": "IiIiU2NvcmluZywgbWV0cmljIGNhbGN1bGF0aW9ucywgYW5kIHBhaXJlZCBxdWVyeS1sZXZlbCBjb21wYXJpc29ucy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0aW1lCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBudW1weSBhcyBucAoKCk1FVFJJQ19OQU1FUyA9ICgicmVjYWxsX2F0XzEiLCAicmVjYWxsX2F0XzUiLCAicmVjYWxsX2F0XzEwIiwgIm1ycl9hdF8xMCIsICJuZGNnX2F0XzEwIikKCgpkZWYgZXZhbHVhdGVfcmV0cmlldmFsKAogICAgcXVlcnlfZW1iZWRkaW5nczogbnAubmRhcnJheSwKICAgIHBhc3NhZ2VfZW1iZWRkaW5nczogbnAubmRhcnJheSwKICAgIHF1ZXJ5X2lkczogbGlzdFtzdHJdLAogICAgY2FuZGlkYXRlX2lkczogbGlzdFtzdHJdLAogICAgcXJlbHM6IGRpY3Rbc3RyLCBkaWN0W3N0ciwgaW50XV0sCiAgICBjb3JwdXM6IGRpY3Rbc3RyLCBkaWN0W3N0ciwgc3RyXV0sCikgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJDb21wdXRlIGNvc2luZS1zY29yZSByYW5raW5ncyBhbmQgc3RhbmRhcmQgQkVJUi1zdHlsZSB0b3AtayBtZXRyaWNzLiIiIgogICAgcXVlcnlfZW1iZWRkaW5ncyA9IG5wLmFzYXJyYXkocXVlcnlfZW1iZWRkaW5ncywgZHR5cGU9bnAuZmxvYXQzMikKICAgIHBhc3NhZ2VfZW1iZWRkaW5ncyA9IG5wLmFzYXJyYXkocGFzc2FnZV9lbWJlZGRpbmdzLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgaWYgcXVlcnlfZW1iZWRkaW5ncy5uZGltICE9IDIgb3IgcGFzc2FnZV9lbWJlZGRpbmdzLm5kaW0gIT0gMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJRdWVyeSBhbmQgcGFzc2FnZSBlbWJlZGRpbmdzIG11c3QgYm90aCBiZSAyRCBhcnJheXMuIikKICAgIGlmIHF1ZXJ5X2VtYmVkZGluZ3Muc2hhcGVbMF0gIT0gbGVuKHF1ZXJ5X2lkcyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUXVlcnkgZW1iZWRkaW5nIGNvdW50IGRvZXMgbm90IG1hdGNoIHF1ZXJ5IElEcy4iKQogICAgaWYgcGFzc2FnZV9lbWJlZGRpbmdzLnNoYXBlWzBdICE9IGxlbihjYW5kaWRhdGVfaWRzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJQYXNzYWdlIGVtYmVkZGluZyBjb3VudCBkb2VzIG5vdCBtYXRjaCBjYW5kaWRhdGUgSURzLiIpCiAgICBpZiBxdWVyeV9lbWJlZGRpbmdzLnNoYXBlWzFdICE9IHBhc3NhZ2VfZW1iZWRkaW5ncy5zaGFwZVsxXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmIkVtYmVkZGluZyBkaW1lbnNpb25zIGRpZmZlcjogcXVlcnk9e3F1ZXJ5X2VtYmVkZGluZ3Muc2hhcGVbMV19LCBwYXNzYWdlPXtwYXNzYWdlX2VtYmVkZGluZ3Muc2hhcGVbMV19LiIKICAgICAgICApCiAgICBpZiBsZW4oc2V0KGNhbmRpZGF0ZV9pZHMpKSAhPSBsZW4oY2FuZGlkYXRlX2lkcyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ2FuZGlkYXRlIGRvY3VtZW50IElEcyBtdXN0IGJlIHVuaXF1ZS4iKQogICAgaWYgYW55KHF1ZXJ5X2lkIG5vdCBpbiBxcmVscyBvciBub3QgcXJlbHNbcXVlcnlfaWRdIGZvciBxdWVyeV9pZCBpbiBxdWVyeV9pZHMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV2ZXJ5IGV2YWx1YXRlZCBxdWVyeSBtdXN0IGhhdmUgYXQgbGVhc3Qgb25lIHBvc2l0aXZlIHFyZWwuIikKICAgIGlmIGFueShub3Qgc2V0KHFyZWxzW3F1ZXJ5X2lkXSkgPD0gc2V0KGNhbmRpZGF0ZV9pZHMpIGZvciBxdWVyeV9pZCBpbiBxdWVyeV9pZHMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRoZSBjYW5kaWRhdGUgcG9vbCBtdXN0IGluY2x1ZGUgYWxsIGp1ZGdlZCByZWxldmFudCBkb2N1bWVudHMgZm9yIGVhY2ggcXVlcnkuIikKICAgIGlmIG5vdCBucC5pc2Zpbml0ZShxdWVyeV9lbWJlZGRpbmdzKS5hbGwoKSBvciBub3QgbnAuaXNmaW5pdGUocGFzc2FnZV9lbWJlZGRpbmdzKS5hbGwoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFbWJlZGRpbmcgbWF0cmljZXMgY29udGFpbiBOYU4gb3IgaW5maW5pdGUgdmFsdWVzLiIpCgogICAgc2NvcmluZ19zdGFydGVkID0gdGltZS5tb25vdG9uaWMoKQogICAgc2ltaWxhcml0aWVzID0gcXVlcnlfZW1iZWRkaW5ncyBAIHBhc3NhZ2VfZW1iZWRkaW5ncy5UCiAgICBzY29yaW5nX3NlY29uZHMgPSB0aW1lLm1vbm90b25pYygpIC0gc2NvcmluZ19zdGFydGVkCiAgICBpZiBzaW1pbGFyaXRpZXMuc2hhcGUgIT0gKGxlbihxdWVyeV9pZHMpLCBsZW4oY2FuZGlkYXRlX2lkcykpIG9yIG5vdCBucC5pc2Zpbml0ZShzaW1pbGFyaXRpZXMpLmFsbCgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNpbWlsYXJpdHkgbWF0cml4IGhhcyBhbiBpbnZhbGlkIHNoYXBlIG9yIG5vbi1maW5pdGUgdmFsdWVzLiIpCgogICAgcGVyX3F1ZXJ5ID0gW10KICAgIG1ldHJpY190b3RhbHMgPSB7bmFtZTogMC4wIGZvciBuYW1lIGluIE1FVFJJQ19OQU1FU30KICAgIGZvciByb3dfaW5kZXgsIHF1ZXJ5X2lkIGluIGVudW1lcmF0ZShxdWVyeV9pZHMpOgogICAgICAgIG9yZGVyID0gbnAuYXJnc29ydCgtc2ltaWxhcml0aWVzW3Jvd19pbmRleF0sIGtpbmQ9InN0YWJsZSIpCiAgICAgICAgcmFua2VkX2lkcyA9IFtjYW5kaWRhdGVfaWRzW2luZGV4XSBmb3IgaW5kZXggaW4gb3JkZXJdCiAgICAgICAgcmVsZXZhbmNlID0gcXJlbHNbcXVlcnlfaWRdCiAgICAgICAgcXVlcnlfbWV0cmljcyA9IF9tZXRyaWNzX2Zvcl9yYW5raW5nKHJhbmtlZF9pZHMsIHJlbGV2YW5jZSkKICAgICAgICBmb3IgbWV0cmljX25hbWUgaW4gTUVUUklDX05BTUVTOgogICAgICAgICAgICBtZXRyaWNfdG90YWxzW21ldHJpY19uYW1lXSArPSBxdWVyeV9tZXRyaWNzW21ldHJpY19uYW1lXQoKICAgICAgICB0b3BfMTAgPSBbXQogICAgICAgIGZvciByYW5rLCBjYW5kaWRhdGVfaW5kZXggaW4gZW51bWVyYXRlKG9yZGVyWzoxMF0sIHN0YXJ0PTEpOgogICAgICAgICAgICBkb2N1bWVudF9pZCA9IGNhbmRpZGF0ZV9pZHNbY2FuZGlkYXRlX2luZGV4XQogICAgICAgICAgICBkb2N1bWVudCA9IGNvcnB1c1tkb2N1bWVudF9pZF0KICAgICAgICAgICAgdG9wXzEwLmFwcGVuZCgKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAiZG9jdW1lbnRfaWQiOiBkb2N1bWVudF9pZCwKICAgICAgICAgICAgICAgICAgICAicmFuayI6IHJhbmssCiAgICAgICAgICAgICAgICAgICAgInNjb3JlIjogZmxvYXQoc2ltaWxhcml0aWVzW3Jvd19pbmRleCwgY2FuZGlkYXRlX2luZGV4XSksCiAgICAgICAgICAgICAgICAgICAgImlzX3JlbGV2YW50IjogZG9jdW1lbnRfaWQgaW4gcmVsZXZhbmNlLAogICAgICAgICAgICAgICAgICAgICJyZWxldmFuY2VfZ3JhZGUiOiByZWxldmFuY2UuZ2V0KGRvY3VtZW50X2lkLCAwKSwKICAgICAgICAgICAgICAgICAgICAicGFzc2FnZV90ZXh0IjogX3Bhc3NhZ2VfdGV4dChkb2N1bWVudCksCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKICAgICAgICBwZXJfcXVlcnkuYXBwZW5kKAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAicXVlcnlfaWQiOiBxdWVyeV9pZCwKICAgICAgICAgICAgICAgICJtZXRyaWNzIjogcXVlcnlfbWV0cmljcywKICAgICAgICAgICAgICAgICJnb2xkX3JlbGV2YW50X2RvY3VtZW50X2lkcyI6IHNvcnRlZChyZWxldmFuY2UsIGtleT1fc3RhYmxlX2lkX2tleSksCiAgICAgICAgICAgICAgICAidG9wXzEwIjogdG9wXzEwLAogICAgICAgICAgICB9CiAgICAgICAgKQoKICAgIGNvdW50ID0gbGVuKHF1ZXJ5X2lkcykKICAgIHJldHVybiB7CiAgICAgICAgIm1ldHJpY3MiOiB7bmFtZTogbWV0cmljX3RvdGFsc1tuYW1lXSAvIGNvdW50IGZvciBuYW1lIGluIE1FVFJJQ19OQU1FU30sCiAgICAgICAgInBlcl9xdWVyeSI6IHBlcl9xdWVyeSwKICAgICAgICAic2NvcmluZ19zZWNvbmRzIjogc2NvcmluZ19zZWNvbmRzLAogICAgICAgICJzY29yZWRfcGFpcnMiOiBsZW4ocXVlcnlfaWRzKSAqIGxlbihjYW5kaWRhdGVfaWRzKSwKICAgICAgICAic2ltaWxhcml0eV9mdW5jdGlvbiI6ICJkb3RfcHJvZHVjdF9vZl9sMl9ub3JtYWxpemVkX2VtYmVkZGluZ3MgKGNvc2luZSBzaW1pbGFyaXR5KSIsCiAgICB9CgoKZGVmIGNvbWJpbmVfcXVlcnlfcmVzdWx0cygKICAgIHF1ZXJ5X2lkczogbGlzdFtzdHJdLAogICAgcXVlcmllczogZGljdFtzdHIsIHN0cl0sCiAgICBxcmVsczogZGljdFtzdHIsIGRpY3Rbc3RyLCBpbnRdXSwKICAgIGJhc2VsaW5lX3Jlc3VsdDogZGljdFtzdHIsIEFueV0sCiAgICBrcHJfcmVzdWx0OiBkaWN0W3N0ciwgQW55XSwKKSAtPiB0dXBsZVtsaXN0W2RpY3Rbc3RyLCBBbnldXSwgZGljdFtzdHIsIEFueV1dOgogICAgIiIiQ3JlYXRlIGRldGFpbGVkIHBhaXJlZCBwcmVkaWN0aW9ucyBhbmQgdXAgdG8gZml2ZSBleGFtcGxlcyBwZXIgY2F0ZWdvcnkuIiIiCiAgICBiYXNlbGluZV9yb3dzID0ge3Jvd1sicXVlcnlfaWQiXTogcm93IGZvciByb3cgaW4gYmFzZWxpbmVfcmVzdWx0WyJwZXJfcXVlcnkiXX0KICAgIGtwcl9yb3dzID0ge3Jvd1sicXVlcnlfaWQiXTogcm93IGZvciByb3cgaW4ga3ByX3Jlc3VsdFsicGVyX3F1ZXJ5Il19CiAgICBjb21iaW5lZCA9IFtdCiAgICBmb3IgcXVlcnlfaWQgaW4gcXVlcnlfaWRzOgogICAgICAgIGJhc2VsaW5lID0gYmFzZWxpbmVfcm93c1txdWVyeV9pZF0KICAgICAgICBrcHIgPSBrcHJfcm93c1txdWVyeV9pZF0KICAgICAgICBiYXNlbGluZV9uZGNnID0gYmFzZWxpbmVbIm1ldHJpY3MiXVsibmRjZ19hdF8xMCJdCiAgICAgICAga3ByX25kY2cgPSBrcHJbIm1ldHJpY3MiXVsibmRjZ19hdF8xMCJdCiAgICAgICAgaWYga3ByX25kY2cgPiBiYXNlbGluZV9uZGNnICsgMWUtMTI6CiAgICAgICAgICAgIGRpcmVjdGlvbiA9ICJpbXByb3ZlZCIKICAgICAgICBlbGlmIGtwcl9uZGNnIDwgYmFzZWxpbmVfbmRjZyAtIDFlLTEyOgogICAgICAgICAgICBkaXJlY3Rpb24gPSAid29yc2VuZWQiCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZGlyZWN0aW9uID0gInRpZSIKICAgICAgICBjb21iaW5lZC5hcHBlbmQoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJxdWVyeV9pZCI6IHF1ZXJ5X2lkLAogICAgICAgICAgICAgICAgInF1ZXJ5X3RleHQiOiBxdWVyaWVzW3F1ZXJ5X2lkXSwKICAgICAgICAgICAgICAgICJnb2xkX3JlbGV2YW50X2RvY3VtZW50X2lkcyI6IHNvcnRlZChxcmVsc1txdWVyeV9pZF0sIGtleT1fc3RhYmxlX2lkX2tleSksCiAgICAgICAgICAgICAgICAiYmFzZWxpbmUiOiBiYXNlbGluZSwKICAgICAgICAgICAgICAgICJrcHIiOiBrcHIsCiAgICAgICAgICAgICAgICAiY29tcGFyaXNvbiI6IHsKICAgICAgICAgICAgICAgICAgICAicmFua2luZ19jaGFuZ2VfYnlfbmRjZ19hdF8xMCI6IGRpcmVjdGlvbiwKICAgICAgICAgICAgICAgICAgICAibmRjZ19hdF8xMF9kZWx0YV9rcHJfbWludXNfYmFzZWxpbmUiOiBrcHJfbmRjZyAtIGJhc2VsaW5lX25kY2csCiAgICAgICAgICAgICAgICAgICAgIm1ycl9hdF8xMF9kZWx0YV9rcHJfbWludXNfYmFzZWxpbmUiOiAoCiAgICAgICAgICAgICAgICAgICAgICAgIGtwclsibWV0cmljcyJdWyJtcnJfYXRfMTAiXSAtIGJhc2VsaW5lWyJtZXRyaWNzIl1bIm1ycl9hdF8xMCJdCiAgICAgICAgICAgICAgICAgICAgKSwKICAgICAgICAgICAgICAgICAgICAicmVjYWxsX2F0XzEwX2RlbHRhX2twcl9taW51c19iYXNlbGluZSI6ICgKICAgICAgICAgICAgICAgICAgICAgICAga3ByWyJtZXRyaWNzIl1bInJlY2FsbF9hdF8xMCJdIC0gYmFzZWxpbmVbIm1ldHJpY3MiXVsicmVjYWxsX2F0XzEwIl0KICAgICAgICAgICAgICAgICAgICApLAogICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgfQogICAgICAgICkKCiAgICBrcHJfYmV0dGVyID0gW3JvdyBmb3Igcm93IGluIGNvbWJpbmVkIGlmIHJvd1siY29tcGFyaXNvbiJdWyJyYW5raW5nX2NoYW5nZV9ieV9uZGNnX2F0XzEwIl0gPT0gImltcHJvdmVkIl0KICAgIGJhc2VsaW5lX2JldHRlciA9IFtyb3cgZm9yIHJvdyBpbiBjb21iaW5lZCBpZiByb3dbImNvbXBhcmlzb24iXVsicmFua2luZ19jaGFuZ2VfYnlfbmRjZ19hdF8xMCJdID09ICJ3b3JzZW5lZCJdCiAgICBib3RoX2ZhaWxlZCA9IFsKICAgICAgICByb3cKICAgICAgICBmb3Igcm93IGluIGNvbWJpbmVkCiAgICAgICAgaWYgcm93WyJiYXNlbGluZSJdWyJtZXRyaWNzIl1bInJlY2FsbF9hdF8xMCJdID09IDAuMAogICAgICAgIGFuZCByb3dbImtwciJdWyJtZXRyaWNzIl1bInJlY2FsbF9hdF8xMCJdID09IDAuMAogICAgXQogICAga3ByX2JldHRlci5zb3J0KGtleT1sYW1iZGEgcm93OiByb3dbImNvbXBhcmlzb24iXVsibmRjZ19hdF8xMF9kZWx0YV9rcHJfbWludXNfYmFzZWxpbmUiXSwgcmV2ZXJzZT1UcnVlKQogICAgYmFzZWxpbmVfYmV0dGVyLnNvcnQoa2V5PWxhbWJkYSByb3c6IHJvd1siY29tcGFyaXNvbiJdWyJuZGNnX2F0XzEwX2RlbHRhX2twcl9taW51c19iYXNlbGluZSJdKQogICAgYm90aF9mYWlsZWQuc29ydChrZXk9bGFtYmRhIHJvdzogX3N0YWJsZV9pZF9rZXkocm93WyJxdWVyeV9pZCJdKSkKCiAgICBkZWYgZmFpbHVyZV9jYXNlKHJvdzogZGljdFtzdHIsIEFueV0sIGNhc2VfdHlwZTogc3RyKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAiY2FzZV90eXBlIjogY2FzZV90eXBlLAogICAgICAgICAgICAicXVlcnlfaWQiOiByb3dbInF1ZXJ5X2lkIl0sCiAgICAgICAgICAgICJxdWVyeV90ZXh0Ijogcm93WyJxdWVyeV90ZXh0Il0sCiAgICAgICAgICAgICJnb2xkX3JlbGV2YW50X2RvY3VtZW50X2lkcyI6IHJvd1siZ29sZF9yZWxldmFudF9kb2N1bWVudF9pZHMiXSwKICAgICAgICAgICAgImNhdGVnb3J5IjogInVuY2xhc3NpZmllZCIsCiAgICAgICAgICAgICJ0ZW50YXRpdmVfb2JzZXJ2YXRpb24iOiAoCiAgICAgICAgICAgICAgICAiVGhlIHBhaXJlZCB0b3AtMTAgcmFua2luZ3MgYW5kIFNjaUZhY3QgcXJlbHMgYXJlIGluY2x1ZGVkIGZvciBtYW51YWwgaW5zcGVjdGlvbjsgIgogICAgICAgICAgICAgICAgIm5vIGNhdXNhbCBmYWlsdXJlIGNhdGVnb3J5IGlzIGluZmVycmVkIGZyb20gcmVsZXZhbmNlIGxhYmVscyBhbG9uZS4iCiAgICAgICAgICAgICksCiAgICAgICAgICAgICJiYXNlbGluZV90b3BfMTAiOiByb3dbImJhc2VsaW5lIl1bInRvcF8xMCJdLAogICAgICAgICAgICAia3ByX3RvcF8xMCI6IHJvd1sia3ByIl1bInRvcF8xMCJdLAogICAgICAgICAgICAiYmFzZWxpbmVfbWV0cmljcyI6IHJvd1siYmFzZWxpbmUiXVsibWV0cmljcyJdLAogICAgICAgICAgICAia3ByX21ldHJpY3MiOiByb3dbImtwciJdWyJtZXRyaWNzIl0sCiAgICAgICAgfQoKICAgIGNhdGVnb3JpZXMgPSB7CiAgICAgICAgImtwcl9iZXR0ZXIiOiBbZmFpbHVyZV9jYXNlKHJvdywgImtwcl9iZXR0ZXJfYnlfbmRjZ19hdF8xMCIpIGZvciByb3cgaW4ga3ByX2JldHRlcls6NV1dLAogICAgICAgICJiYXNlbGluZV9iZXR0ZXIiOiBbZmFpbHVyZV9jYXNlKHJvdywgImJhc2VsaW5lX2JldHRlcl9ieV9uZGNnX2F0XzEwIikgZm9yIHJvdyBpbiBiYXNlbGluZV9iZXR0ZXJbOjVdXSwKICAgICAgICAiYm90aF9mYWlsZWQiOiBbZmFpbHVyZV9jYXNlKHJvdywgIm5laXRoZXJfcmV0cmlldmVkX3JlbGV2YW50X2luX3RvcF8xMCIpIGZvciByb3cgaW4gYm90aF9mYWlsZWRbOjVdXSwKICAgIH0KICAgIGNvdW50cyA9IHsKICAgICAgICAia3ByX2JldHRlcl9hdmFpbGFibGUiOiBsZW4oa3ByX2JldHRlciksCiAgICAgICAgImJhc2VsaW5lX2JldHRlcl9hdmFpbGFibGUiOiBsZW4oYmFzZWxpbmVfYmV0dGVyKSwKICAgICAgICAiYm90aF9mYWlsZWRfYXZhaWxhYmxlIjogbGVuKGJvdGhfZmFpbGVkKSwKICAgICAgICAiZXhhbXBsZXNfcmV0dXJuZWRfcGVyX2NhdGVnb3J5IjogNSwKICAgICAgICAic2VsZWN0aW9uX21ldHJpYyI6ICJwZXItcXVlcnkgbkRDR0AxMDsgYm90aF9mYWlsZWQgbWVhbnMgemVybyBSZWNhbGxAMTAgZm9yIGJvdGggbW9kZWxzIiwKICAgIH0KICAgIHJldHVybiBjb21iaW5lZCwgeyoqY2F0ZWdvcmllcywgImNvdW50cyI6IGNvdW50c30KCgpkZWYgbWV0cmljX2RlbHRhcyhiYXNlbGluZV9tZXRyaWNzOiBkaWN0W3N0ciwgZmxvYXRdLCBrcHJfbWV0cmljczogZGljdFtzdHIsIGZsb2F0XSkgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgIHJldHVybiB7bmFtZTogZmxvYXQoa3ByX21ldHJpY3NbbmFtZV0gLSBiYXNlbGluZV9tZXRyaWNzW25hbWVdKSBmb3IgbmFtZSBpbiBNRVRSSUNfTkFNRVN9CgoKZGVmIF9tZXRyaWNzX2Zvcl9yYW5raW5nKHJhbmtlZF9pZHM6IGxpc3Rbc3RyXSwgcmVsZXZhbmNlOiBkaWN0W3N0ciwgaW50XSkgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgIHJlbGV2YW50X2lkcyA9IHNldChyZWxldmFuY2UpCiAgICByZWxldmFudF9jb3VudCA9IGxlbihyZWxldmFudF9pZHMpCiAgICBpZiByZWxldmFudF9jb3VudCA9PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk1ldHJpY3MgYXJlIHVuZGVmaW5lZCBmb3IgYSBxdWVyeSB3aXRob3V0IGp1ZGdlZCByZWxldmFudCBkb2N1bWVudHMuIikKICAgIG1ldHJpY3MgPSB7fQogICAgZm9yIGsgaW4gKDEsIDUsIDEwKToKICAgICAgICBtZXRyaWNzW2YicmVjYWxsX2F0X3trfSJdID0gbGVuKHNldChyYW5rZWRfaWRzWzprXSkgJiByZWxldmFudF9pZHMpIC8gcmVsZXZhbnRfY291bnQKCiAgICByZWNpcHJvY2FsX3JhbmsgPSAwLjAKICAgIGZvciByYW5rLCBkb2N1bWVudF9pZCBpbiBlbnVtZXJhdGUocmFua2VkX2lkc1s6MTBdLCBzdGFydD0xKToKICAgICAgICBpZiBkb2N1bWVudF9pZCBpbiByZWxldmFudF9pZHM6CiAgICAgICAgICAgIHJlY2lwcm9jYWxfcmFuayA9IDEuMCAvIHJhbmsKICAgICAgICAgICAgYnJlYWsKICAgIG1ldHJpY3NbIm1ycl9hdF8xMCJdID0gcmVjaXByb2NhbF9yYW5rCgogICAgZGNnID0gMC4wCiAgICBmb3IgcmFuaywgZG9jdW1lbnRfaWQgaW4gZW51bWVyYXRlKHJhbmtlZF9pZHNbOjEwXSwgc3RhcnQ9MSk6CiAgICAgICAgZ3JhZGUgPSByZWxldmFuY2UuZ2V0KGRvY3VtZW50X2lkLCAwKQogICAgICAgIGlmIGdyYWRlID4gMDoKICAgICAgICAgICAgZGNnICs9ICgyKipncmFkZSAtIDEpIC8gbnAubG9nMihyYW5rICsgMSkKICAgIGlkZWFsX2dyYWRlcyA9IHNvcnRlZChyZWxldmFuY2UudmFsdWVzKCksIHJldmVyc2U9VHJ1ZSlbOjEwXQogICAgaWRlYWxfZGNnID0gc3VtKCgyKipncmFkZSAtIDEpIC8gbnAubG9nMihyYW5rICsgMSkgZm9yIHJhbmssIGdyYWRlIGluIGVudW1lcmF0ZShpZGVhbF9ncmFkZXMsIHN0YXJ0PTEpKQogICAgbWV0cmljc1sibmRjZ19hdF8xMCJdID0gZmxvYXQoZGNnIC8gaWRlYWxfZGNnKSBpZiBpZGVhbF9kY2cgPiAwIGVsc2UgMC4wCiAgICByZXR1cm4ge25hbWU6IGZsb2F0KHZhbHVlKSBmb3IgbmFtZSwgdmFsdWUgaW4gbWV0cmljcy5pdGVtcygpfQoKCmRlZiBfcGFzc2FnZV90ZXh0KGRvY3VtZW50OiBkaWN0W3N0ciwgc3RyXSkgLT4gc3RyOgogICAgdGl0bGUgPSBkb2N1bWVudC5nZXQoInRpdGxlIiwgIiIpLnN0cmlwKCkKICAgIHRleHQgPSBkb2N1bWVudC5nZXQoInRleHQiLCAiIikuc3RyaXAoKQogICAgcmV0dXJuIGYie3RpdGxlfVxue3RleHR9IiBpZiB0aXRsZSBhbmQgdGV4dCBlbHNlIHRpdGxlIG9yIHRleHQKCgpkZWYgX3N0YWJsZV9pZF9rZXkodmFsdWU6IHN0cik6CiAgICByZXR1cm4gKDAsIGludCh2YWx1ZSkpIGlmIHZhbHVlLmlzZGlnaXQoKSBlbHNlICgxLCB2YWx1ZSkK", "src/reporting.py": "IiIiU2luZ2xlLWZpbGUgSlNPTiByZXBvcnRpbmcgYW5kIGVudmlyb25tZW50L3Jlc291cmNlIG1ldGFkYXRhIGhlbHBlcnMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaW1wb3J0bGliLm1ldGFkYXRhCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcGxhdGZvcm0KaW1wb3J0IHRpbWUKaW1wb3J0IHRyYWNlYmFjawpmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZXpvbmUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCgpkZWYgYnVpbGRfZW1wdHlfcmVwb3J0KCkgLT4gZGljdFtzdHIsIEFueV06CiAgICByZXR1cm4gewogICAgICAgICJtZXRhZGF0YSI6IHsKICAgICAgICAgICAgInByb2plY3QiOiAiTGlnaHR3ZWlnaHQgS1BSIFJldHJpZXZhbCBFdmFsdWF0aW9uIiwKICAgICAgICAgICAgImNyZWF0ZWRfYXRfdXRjIjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCksCiAgICAgICAgICAgICJwYXBlciI6ICJEeW5hbWljIEluamVjdGlvbiBvZiBFbnRpdHkgS25vd2xlZGdlIGludG8gRGVuc2UgUmV0cmlldmVycyAoRmluZGluZ3MgRU1OTFAgMjAyNSkiLAogICAgICAgICAgICAicGFwZXJfdXJsIjogImh0dHBzOi8vYWNsYW50aG9sb2d5Lm9yZy8yMDI1LmZpbmRpbmdzLWVtbmxwLjkxNS8iLAogICAgICAgICAgICAib2ZmaWNpYWxfcmVwb3NpdG9yeSI6ICJodHRwczovL2dpdGh1Yi5jb20va25vd2xlZGdlYWJsZS1lbWJlZGRpbmcva25vd2xlZGdlYWJsZS1lbWJlZGRpbmciLAogICAgICAgICAgICAia3ByX21vZGVsX2NhcmQiOiAiaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9rbm93bGVkZ2VhYmxlLWFpL2twci1iZ2UtYmFzZS1lbi12MS41IiwKICAgICAgICAgICAgImJhc2VsaW5lX21vZGVsX2NhcmQiOiAiaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9CQUFJL2JnZS1iYXNlLWVuLXYxLjUiLAogICAgICAgIH0sCiAgICAgICAgImNvbmZpZ3VyYXRpb24iOiB7CiAgICAgICAgICAgICJyYW5kb21fc2VlZCI6IDQyLAogICAgICAgICAgICAidGFyZ2V0X3F1ZXJpZXMiOiA0MCwKICAgICAgICAgICAgInRhcmdldF9jYW5kaWRhdGVfcGFzc2FnZXMiOiA4MDAsCiAgICAgICAgICAgICJtYXhfc2VxdWVuY2VfbGVuZ3RoIjogMjU2LAogICAgICAgICAgICAiYmF0Y2hfc2l6ZSI6IDgsCiAgICAgICAgICAgICJzb2Z0X2RlYWRsaW5lX21pbnV0ZXMiOiAyNSwKICAgICAgICAgICAgImhhcmRfZGVhZGxpbmVfbWludXRlcyI6IDMwLAogICAgICAgICAgICAicHJlY2lzaW9uX3BvbGljeSI6ICJmbG9hdDMyIHRvIGtlZXAgYm90aCBtb2RlbHMgY29tcGFyYWJsZSBhbmQgYXZvaWQgS1BSIGVudGl0eS1pbnB1dCBkdHlwZSBtaXNtYXRjaCIsCiAgICAgICAgICAgICJub190cmFpbmluZyI6IFRydWUsCiAgICAgICAgICAgICJwYWlkX2FwaXMiOiBGYWxzZSwKICAgICAgICB9LAogICAgICAgICJlbnZpcm9ubWVudCI6IHt9LAogICAgICAgICJtb2RlbF9pbmZvcm1hdGlvbiI6IHsiYmFzZWxpbmUiOiB7fSwgImtwciI6IHt9fSwKICAgICAgICAiZGF0YXNldCI6IHt9LAogICAgICAgICJldmFsdWF0aW9uX3Byb3RvY29sIjogewogICAgICAgICAgICAiZXZhbHVhdGlvbl90eXBlIjogInJlc3RyaWN0ZWRfY2FuZGlkYXRlX3JldHJpZXZhbCIsCiAgICAgICAgICAgICJtZXRyaWNzIjogWyJSZWNhbGxAMSIsICJSZWNhbGxANSIsICJSZWNhbGxAMTAiLCAiTVJSQDEwIiwgIm5EQ0dAMTAiXSwKICAgICAgICAgICAgIm1ldHJpY19kZWZpbml0aW9ucyI6IHsKICAgICAgICAgICAgICAgICJyZWNhbGxfYXRfayI6ICJGb3IgZWFjaCBxdWVyeSwgcmVsZXZhbnQgZG9jdW1lbnRzIGluIHRvcC1rIGRpdmlkZWQgYnkgYWxsIHBvc2l0aXZlLXFyZWwgZG9jdW1lbnRzIGZvciB0aGF0IHF1ZXJ5OyBtYWNyby1hdmVyYWdlIG92ZXIgcXVlcmllcy4iLAogICAgICAgICAgICAgICAgIm1ycl9hdF8xMCI6ICJNYWNyby1hdmVyYWdlIHJlY2lwcm9jYWwgcmFuayBvZiB0aGUgZmlyc3QgcG9zaXRpdmUtcXJlbCBkb2N1bWVudCBpbiB0b3AgMTA7IHplcm8gaWYgbm9uZSBhcHBlYXJzLiIsCiAgICAgICAgICAgICAgICAibmRjZ19hdF8xMCI6ICJNYWNyby1hdmVyYWdlIERDR0AxMCBkaXZpZGVkIGJ5IGlkZWFsIERDR0AxMDsgZ3JhZGVkIGdhaW4gaXMgMl5yZWxldmFuY2UtMSBhbmQgZGlzY291bnQgaXMgbG9nMihyYW5rKzEpLiIsCiAgICAgICAgICAgIH0sCiAgICAgICAgICAgICJzY29yaW5nIjogImRvdCBwcm9kdWN0IG9mIEwyLW5vcm1hbGl6ZWQgZW1iZWRkaW5ncyAoY29zaW5lIHNpbWlsYXJpdHkpIiwKICAgICAgICAgICAgInF1ZXJ5X2Zvcm1hdHRpbmciOiB7CiAgICAgICAgICAgICAgICAiYmFzZWxpbmUiOiAiQkdFIHJlY29tbWVuZGVkIHJldHJpZXZhbCBwcmVmaXggaXMgcHJlcGVuZGVkIHRvIHF1ZXJpZXM7IHBhc3NhZ2VzIGhhdmUgbm8gcHJlZml4LiIsCiAgICAgICAgICAgICAgICAia3ByIjogIlJhdyBxdWVyeSBhbmQgcGFzc2FnZSB0ZXh0IHBlciB0aGUgS1BSIFNlbnRlbmNlVHJhbnNmb3JtZXIgbW9kZWwtY2FyZCBleGFtcGxlLiIsCiAgICAgICAgICAgIH0sCiAgICAgICAgICAgICJxcmVsX2ludGVycHJldGF0aW9uIjogKAogICAgICAgICAgICAgICAgIk1ldHJpY3MgdXNlIHBvc2l0aXZlIFNjaUZhY3QgdGVzdCBxcmVscyBhcyB0aGUganVkZ2VkIHJlbGV2YW50IHNldC4gVW5qdWRnZWQgY2FuZGlkYXRlICIKICAgICAgICAgICAgICAgICJwYXNzYWdlcyBhcmUgdHJlYXRlZCBhcyBub24tcmVsZXZhbnQgZm9yIHRoaXMgc2FtcGxlZCByYW5raW5nIGNhbGN1bGF0aW9uOyBTY2lGYWN0ICIKICAgICAgICAgICAgICAgICJxcmVscyBhcmUgaW5jb21wbGV0ZSBhbmQgZG8gbm90IGVzdGFibGlzaCBzdXBwb3J0IHZlcnN1cyBjb250cmFkaWN0aW9uIGZvciBlYWNoIGNsYWltLiIKICAgICAgICAgICAgKSwKICAgICAgICAgICAgImNhbmRpZGF0ZV9wb29sX25vdGUiOiAiVGhpcyBpcyBhIGRldGVybWluaXN0aWMgcmVzdHJpY3RlZC1jYW5kaWRhdGUgZXZhbHVhdGlvbiwgbm90IGZ1bGwtY29ycHVzIEJFSVIgb3IgdGhlIHBhcGVyJ3MgYmVuY2htYXJrLiIsCiAgICAgICAgICAgICJwZXJfcXVlcnlfcmFua2luZ19jb21wYXJpc29uIjogIktQUiB2ZXJzdXMgYmFzZWxpbmUgbkRDR0AxMDsgdGllcyBhcmUgcmVjb3JkZWQgZXhwbGljaXRseS4iLAogICAgICAgIH0sCiAgICAgICAgIm1ldHJpY3MiOiB7ImJhc2VsaW5lIjogTm9uZSwgImtwciI6IE5vbmUsICJjb21wYXJpc29uIjogTm9uZX0sCiAgICAgICAgInJlc291cmNlX3VzYWdlIjoge30sCiAgICAgICAgInRpbWluZ3MiOiB7CiAgICAgICAgICAgICJkZXBlbmRlbmN5X2luc3RhbGxfc2Vjb25kcyI6IE5vbmUsCiAgICAgICAgICAgICJyZXBvc2l0b3J5X2luc3BlY3Rpb25fc2Vjb25kcyI6IE5vbmUsCiAgICAgICAgICAgICJkYXRhc2V0X2Rvd25sb2FkX3NlY29uZHMiOiBOb25lLAogICAgICAgICAgICAiZGF0YXNldF9wcmVwYXJhdGlvbl9zZWNvbmRzIjogTm9uZSwKICAgICAgICAgICAgIm1vZGVsX2Rvd25sb2FkX3NlY29uZHMiOiB7ImJhc2VsaW5lIjogTm9uZSwgImtwciI6IE5vbmV9LAogICAgICAgICAgICAibW9kZWxfbG9hZF9zZWNvbmRzIjogeyJiYXNlbGluZSI6IE5vbmUsICJrcHIiOiBOb25lfSwKICAgICAgICAgICAgIndhcm11cF9zZWNvbmRzIjogeyJiYXNlbGluZSI6IE5vbmUsICJrcHIiOiBOb25lfSwKICAgICAgICAgICAgInF1ZXJ5X2VuY29kaW5nX3NlY29uZHMiOiB7ImJhc2VsaW5lIjogTm9uZSwgImtwciI6IE5vbmV9LAogICAgICAgICAgICAicGFzc2FnZV9lbmNvZGluZ19zZWNvbmRzIjogeyJiYXNlbGluZSI6IE5vbmUsICJrcHIiOiBOb25lfSwKICAgICAgICAgICAgInF1ZXJ5X2VuY29kaW5nX3NlY29uZHNfcGVyX3F1ZXJ5IjogeyJiYXNlbGluZSI6IE5vbmUsICJrcHIiOiBOb25lfSwKICAgICAgICAgICAgInNjb3Jpbmdfc2Vjb25kcyI6IHsiYmFzZWxpbmUiOiBOb25lLCAia3ByIjogTm9uZX0sCiAgICAgICAgICAgICJldmFsdWF0aW9uX3NlY29uZHMiOiBOb25lLAogICAgICAgICAgICAicmVwb3J0X2V4cG9ydF9zZWNvbmRzIjogTm9uZSwKICAgICAgICAgICAgImVsYXBzZWRfc2Vjb25kcyI6IDAuMCwKICAgICAgICB9LAogICAgICAgICJwZXJfcXVlcnlfcmVzdWx0cyI6IFtdLAogICAgICAgICJmYWlsdXJlX2FuYWx5c2lzIjogeyJrcHJfYmV0dGVyIjogW10sICJiYXNlbGluZV9iZXR0ZXIiOiBbXSwgImJvdGhfZmFpbGVkIjogW10sICJjb3VudHMiOiB7fX0sCiAgICAgICAgInJlc2VhcmNoX3N1bW1hcnkiOiB7CiAgICAgICAgICAgICJvYnNlcnZhdGlvbnMiOiBbXSwKICAgICAgICAgICAgImxpbWl0YXRpb25zIjogWwogICAgICAgICAgICAgICAgIk5vIHJlc3VsdCBmcm9tIHRoaXMgc2FtcGxlZCByZXN0cmljdGVkIGNhbmRpZGF0ZSBzZXQgc2hvdWxkIGJlIHRyZWF0ZWQgYXMgYSBmdWxsIEJFSVIgb3IgcGFwZXIgcmVwcm9kdWN0aW9uLiIsCiAgICAgICAgICAgICAgICAiU2NpRmFjdCByZXRyaWV2YWwgcXJlbHMgZG8gbm90IGxhYmVsIGZhY3R1YWwgc3VwcG9ydCBvciBjb250cmFkaWN0aW9uIGZvciBldmVyeSByZXRyaWV2ZWQgcGFzc2FnZS4iLAogICAgICAgICAgICBdLAogICAgICAgICAgICAicG90ZW50aWFsX3Jlc2VhcmNoX3F1ZXN0aW9ucyI6IFsKICAgICAgICAgICAgICAgICJEb2VzIHRoZSBlZmZlY3QgcGVyc2lzdCBvbiB0aGUgZnVsbCBTY2lGYWN0IGNvcnB1cyBhbmQgb3RoZXIgZW50aXR5LWhlYXZ5IHJldHJpZXZhbCBkYXRhc2V0cz8iLAogICAgICAgICAgICAgICAgIldoaWNoIHF1ZXJ5IHR5cGVzIGJlbmVmaXQgZnJvbSBlbnRpdHkgaW5qZWN0aW9uIHdoZW4ganVkZ21lbnRzIGFyZSBjb21wbGV0ZSBhbmQgc3RyYXRpZmllZD8iLAogICAgICAgICAgICAgICAgIkhvdyBkbyBtZW1vcnktbWFwcGVkIGVudGl0eSBsb29rdXAgYW5kIGVudGl0eS1saW5raW5nIENQVSBjb3N0cyBhZmZlY3QgcHJvZHVjdGlvbiBsYXRlbmN5PyIsCiAgICAgICAgICAgIF0sCiAgICAgICAgICAgICJpbnRlcnByZXRhdGlvbl92aSI6ICJDaMawYSBjw7Mga+G6v3QgcXXhuqMgxJFvIMSR4buDIGvhur90IGx14bqtbiBLUFIgdOG7kXQgaMahbiBCR0UuIMSQw6J5IGzDoCDEkcOhbmggZ2nDoSBzYW1wbGVkIHRyw6puIGNhbmRpZGF0ZSBwb29sIGdp4bubaSBo4bqhbiwga2jDtG5nIHBo4bqjaSBiZW5jaG1hcmsgxJHhuqd5IMSR4bunLiIsCiAgICAgICAgfSwKICAgICAgICAiZXhlY3V0aW9uIjogewogICAgICAgICAgICAic3RhdHVzIjogImZhaWxlZCIsCiAgICAgICAgICAgICJjb21wbGV0ZWRfc3RlcHMiOiBbXSwKICAgICAgICAgICAgImVycm9ycyI6IFtdLAogICAgICAgICAgICAid2FybmluZ3MiOiBbXSwKICAgICAgICAgICAgInNvZnRfZGVhZGxpbmVfcmVhY2hlZCI6IEZhbHNlLAogICAgICAgICAgICAiaGFyZF9kZWFkbGluZV9zZWNvbmRzIjogMTgwMCwKICAgICAgICB9LAogICAgfQoKCmRlZiBzYXZlX3JlcG9ydChyZXBvcnQ6IGRpY3Rbc3RyLCBBbnldLCBvdXRwdXRfcGF0aDogc3RyIHwgUGF0aCkgLT4gUGF0aDoKICAgICIiIkF0b21pY2FsbHkgd3JpdGUgYW5kIHBhcnNlLXZlcmlmeSB0aGUgb25lIG1haW4gSlNPTiByZXBvcnQuIiIiCiAgICBvdXRwdXRfcGF0aCA9IFBhdGgob3V0cHV0X3BhdGgpCiAgICBvdXRwdXRfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgdGVtcG9yYXJ5X3BhdGggPSBvdXRwdXRfcGF0aC53aXRoX25hbWUob3V0cHV0X3BhdGgubmFtZSArICIudG1wIikKICAgIHdpdGggdGVtcG9yYXJ5X3BhdGgub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIHN0cmVhbToKICAgICAgICBqc29uLmR1bXAocmVwb3J0LCBzdHJlYW0sIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIsIGFsbG93X25hbj1GYWxzZSkKICAgICAgICBzdHJlYW0ud3JpdGUoIlxuIikKICAgIHRlbXBvcmFyeV9wYXRoLnJlcGxhY2Uob3V0cHV0X3BhdGgpCiAgICB3aXRoIG91dHB1dF9wYXRoLm9wZW4oInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBzdHJlYW06CiAgICAgICAganNvbi5sb2FkKHN0cmVhbSkKICAgIHJldHVybiBvdXRwdXRfcGF0aAoKCmRlZiByZWNvcmRfZXhjZXB0aW9uKHJlcG9ydDogZGljdFtzdHIsIEFueV0sIHN0YWdlOiBzdHIsIGVycm9yOiBCYXNlRXhjZXB0aW9uKSAtPiBOb25lOgogICAgdHJhY2VfbGluZXMgPSB0cmFjZWJhY2suZm9ybWF0X2V4Y2VwdGlvbih0eXBlKGVycm9yKSwgZXJyb3IsIGVycm9yLl9fdHJhY2ViYWNrX18pCiAgICByZXBvcnRbImV4ZWN1dGlvbiJdWyJlcnJvcnMiXS5hcHBlbmQoCiAgICAgICAgewogICAgICAgICAgICAic3RhZ2UiOiBzdGFnZSwKICAgICAgICAgICAgInR5cGUiOiB0eXBlKGVycm9yKS5fX25hbWVfXywKICAgICAgICAgICAgIm1lc3NhZ2UiOiBzdHIoZXJyb3IpLAogICAgICAgICAgICAidHJhY2ViYWNrX3N1bW1hcnkiOiAiIi5qb2luKHRyYWNlX2xpbmVzWy04Ol0pWy01MDAwOl0sCiAgICAgICAgfQogICAgKQoKCmRlZiBjb2xsZWN0X2Vudmlyb25tZW50KCosIGluY2x1ZGVfdG9yY2g6IGJvb2wgPSBUcnVlKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIGVudmlyb25tZW50ID0gewogICAgICAgICJweXRob25fdmVyc2lvbiI6IHBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCksCiAgICAgICAgInBsYXRmb3JtIjogcGxhdGZvcm0ucGxhdGZvcm0oKSwKICAgICAgICAid29ya2luZ19kaXJlY3RvcnkiOiBvcy5nZXRjd2QoKSwKICAgICAgICAicGFja2FnZV92ZXJzaW9ucyI6IHt9LAogICAgfQogICAgcGFja2FnZV9uYW1lcyA9ICgKICAgICAgICAic2VudGVuY2UtdHJhbnNmb3JtZXJzIiwKICAgICAgICAidHJhbnNmb3JtZXJzIiwKICAgICAgICAiaHVnZ2luZ2ZhY2UtaHViIiwKICAgICAgICAic2FmZXRlbnNvcnMiLAogICAgICAgICJudW1weSIsCiAgICAgICAgInJlcXVlc3RzIiwKICAgICAgICAicHN1dGlsIiwKICAgICAgICAgICAgInB5YXJyb3ciLAogICAgICAgICAgICAicGFuZGFzIiwKICAgICAgICAgICAgInNwYWN5IiwKICAgICAgICAgICAgIm1hcmlzYS10cmllIiwKICAgICkgKyAoKCJ0b3JjaCIsKSBpZiBpbmNsdWRlX3RvcmNoIGVsc2UgKCkpCiAgICBmb3IgcGFja2FnZSBpbiBwYWNrYWdlX25hbWVzOgogICAgICAgIHRyeToKICAgICAgICAgICAgZW52aXJvbm1lbnRbInBhY2thZ2VfdmVyc2lvbnMiXVtwYWNrYWdlXSA9IGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKHBhY2thZ2UpCiAgICAgICAgZXhjZXB0IGltcG9ydGxpYi5tZXRhZGF0YS5QYWNrYWdlTm90Rm91bmRFcnJvcjoKICAgICAgICAgICAgZW52aXJvbm1lbnRbInBhY2thZ2VfdmVyc2lvbnMiXVtwYWNrYWdlXSA9IE5vbmUKCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHBzdXRpbAoKICAgICAgICBtZW1vcnkgPSBwc3V0aWwudmlydHVhbF9tZW1vcnkoKQogICAgICAgIGVudmlyb25tZW50WyJzeXN0ZW1fbWVtb3J5X2J5dGVzIl0gPSB7InRvdGFsIjogbWVtb3J5LnRvdGFsLCAiYXZhaWxhYmxlIjogbWVtb3J5LmF2YWlsYWJsZX0KICAgICAgICBlbnZpcm9ubWVudFsicHJvY2Vzc19yc3NfYnl0ZXMiXSA9IHBzdXRpbC5Qcm9jZXNzKCkubWVtb3J5X2luZm8oKS5yc3MKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGVudmlyb25tZW50WyJzeXN0ZW1fbWVtb3J5X2J5dGVzIl0gPSBOb25lCiAgICAgICAgZW52aXJvbm1lbnRbInByb2Nlc3NfcnNzX2J5dGVzIl0gPSBOb25lCiAgICAgICAgZW52aXJvbm1lbnRbIm1lbW9yeV9wcm9iZV9lcnJvciJdID0gZiJ7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30iCgogICAgaWYgaW5jbHVkZV90b3JjaDoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGltcG9ydCB0b3JjaAoKICAgICAgICAgICAgZW52aXJvbm1lbnRbImN1ZGFfYXZhaWxhYmxlIl0gPSBib29sKHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkpCiAgICAgICAgICAgIGVudmlyb25tZW50WyJjdWRhX2RldmljZV9jb3VudCJdID0gdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKQogICAgICAgICAgICBlbnZpcm9ubWVudFsiY3VkYV9kZXZpY2VzIl0gPSBbdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoaW5kZXgpIGZvciBpbmRleCBpbiByYW5nZSh0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpKV0KICAgICAgICAgICAgZW52aXJvbm1lbnRbInRvcmNoX3ZlcnNpb24iXSA9IHRvcmNoLl9fdmVyc2lvbl9fCiAgICAgICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgICAgICBlbnZpcm9ubWVudFsiY3VkYV92ZXJzaW9uIl0gPSB0b3JjaC52ZXJzaW9uLmN1ZGEKICAgICAgICAgICAgICAgIGVudmlyb25tZW50WyJncHVfbWVtb3J5X2J5dGVzIl0gPSB7CiAgICAgICAgICAgICAgICAgICAgInRvdGFsIjogdG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoMCkudG90YWxfbWVtb3J5LAogICAgICAgICAgICAgICAgICAgICJmcmVlIjogdG9yY2guY3VkYS5tZW1fZ2V0X2luZm8oMClbMF0sCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBlbnZpcm9ubWVudFsiZ3B1X21lbW9yeV9ieXRlcyJdID0gTm9uZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICBlbnZpcm9ubWVudFsiY3VkYV9hdmFpbGFibGUiXSA9IEZhbHNlCiAgICAgICAgICAgIGVudmlyb25tZW50WyJncHVfcHJvYmVfZXJyb3IiXSA9IGYie3R5cGUoZXhjKS5fX25hbWVfX306IHtleGN9IgogICAgZWxzZToKICAgICAgICBlbnZpcm9ubWVudFsiY3VkYV9hdmFpbGFibGUiXSA9IE5vbmUKICAgICAgICBlbnZpcm9ubWVudFsiY3VkYV9kZXZpY2VfY291bnQiXSA9IE5vbmUKICAgICAgICBlbnZpcm9ubWVudFsiZ3B1X21lbW9yeV9ieXRlcyJdID0gTm9uZQogICAgcmV0dXJuIGVudmlyb25tZW50CgoKZGVmIGNvbGxlY3RfcnVudGltZV9yZXNvdXJjZXMoZGV2aWNlOiBzdHIgfCBOb25lID0gTm9uZSkgLT4gZGljdFtzdHIsIEFueV06CiAgICBzbmFwc2hvdDogZGljdFtzdHIsIEFueV0gPSB7ImRldmljZSI6IGRldmljZX0KICAgIHRyeToKICAgICAgICBpbXBvcnQgcHN1dGlsCgogICAgICAgIHZtID0gcHN1dGlsLnZpcnR1YWxfbWVtb3J5KCkKICAgICAgICBzbmFwc2hvdFsicmFtX2J5dGVzIl0gPSB7InRvdGFsIjogdm0udG90YWwsICJhdmFpbGFibGUiOiB2bS5hdmFpbGFibGV9CiAgICAgICAgc25hcHNob3RbInByb2Nlc3NfcnNzX2J5dGVzIl0gPSBwc3V0aWwuUHJvY2VzcygpLm1lbW9yeV9pbmZvKCkucnNzCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBzbmFwc2hvdFsicmFtX3Byb2JlX2Vycm9yIl0gPSBmInt0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfSIKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKCiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgc25hcHNob3RbImdwdV9tZW1vcnlfYnl0ZXMiXSA9IHsKICAgICAgICAgICAgICAgICJ0b3RhbCI6IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9wcm9wZXJ0aWVzKDApLnRvdGFsX21lbW9yeSwKICAgICAgICAgICAgICAgICJmcmVlIjogdG9yY2guY3VkYS5tZW1fZ2V0X2luZm8oMClbMF0sCiAgICAgICAgICAgICAgICAiYWxsb2NhdGVkIjogdG9yY2guY3VkYS5tZW1vcnlfYWxsb2NhdGVkKDApLAogICAgICAgICAgICAgICAgInJlc2VydmVkIjogdG9yY2guY3VkYS5tZW1vcnlfcmVzZXJ2ZWQoMCksCiAgICAgICAgICAgICAgICAicGVha19hbGxvY2F0ZWQiOiB0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKDApLAogICAgICAgICAgICAgICAgInBlYWtfcmVzZXJ2ZWQiOiB0b3JjaC5jdWRhLm1heF9tZW1vcnlfcmVzZXJ2ZWQoMCksCiAgICAgICAgICAgIH0KICAgICAgICBlbHNlOgogICAgICAgICAgICBzbmFwc2hvdFsiZ3B1X21lbW9yeV9ieXRlcyJdID0gTm9uZQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgc25hcHNob3RbImdwdV9wcm9iZV9lcnJvciJdID0gZiJ7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30iCiAgICByZXR1cm4gc25hcHNob3QKCgpkZWYgdXBkYXRlX3Jlc2VhcmNoX3N1bW1hcnkocmVwb3J0OiBkaWN0W3N0ciwgQW55XSkgLT4gTm9uZToKICAgIGJhc2VsaW5lID0gcmVwb3J0LmdldCgibWV0cmljcyIsIHt9KS5nZXQoImJhc2VsaW5lIikKICAgIGtwciA9IHJlcG9ydC5nZXQoIm1ldHJpY3MiLCB7fSkuZ2V0KCJrcHIiKQogICAgY29tcGFyaXNvbiA9IHJlcG9ydC5nZXQoIm1ldHJpY3MiLCB7fSkuZ2V0KCJjb21wYXJpc29uIikKICAgIGlmIG5vdCBpc2luc3RhbmNlKGJhc2VsaW5lLCBkaWN0KSBvciBub3QgaXNpbnN0YW5jZShrcHIsIGRpY3QpIG9yIG5vdCBpc2luc3RhbmNlKGNvbXBhcmlzb24sIGRpY3QpOgogICAgICAgIHJldHVybgogICAgZGVsdGFzID0gY29tcGFyaXNvbi5nZXQoImRlbHRhX2twcl9taW51c19iYXNlbGluZSIsIHt9KQogICAgaWYgcmVwb3J0LmdldCgiZGF0YXNldCIsIHt9KS5nZXQoImV2YWx1YXRpb25fdHlwZSIpID09ICJkaWFnbm9zdGljX29ubHkiOgogICAgICAgIHJlcG9ydFsicmVzZWFyY2hfc3VtbWFyeSJdWyJvYnNlcnZhdGlvbnMiXSA9IFsKICAgICAgICAgICAgIkNo4buJIGNo4bqheSBoYW5kLWF1dGhvcmVkIGRpYWdub3N0aWMgZml4dHVyZSB2w6wga2jDtG5nIHThuqNpIMSRxrDhu6NjIG5ndeG7k24gU2NpRmFjdCBjw7MgbmjDo247IGPDoWMgc+G7kSDEkW8gc2F1IMSRw6J5IGNo4buJIGtp4buDbSB0cmEgbHXhu5NuZyBpbmZlcmVuY2UgdsOgIG1ldHJpYy4iLAogICAgICAgICAgICBmIkZpeHR1cmUgZGlhZ25vc3RpYyBjw7Mge3JlcG9ydC5nZXQoJ2RhdGFzZXQnLCB7fSkuZ2V0KCdzZWxlY3RlZF9xdWVyeV9jb3VudCcsIDApfSBxdWVyeSB2w6AgIgogICAgICAgICAgICBmIntyZXBvcnQuZ2V0KCdkYXRhc2V0Jywge30pLmdldCgnY2FuZGlkYXRlX3Bhc3NhZ2VfY291bnQnLCAwKX0gcGFzc2FnZS4iLAogICAgICAgICAgICAiS2jDtG5nIGRp4buFbiBnaeG6o2kgY8OhYyBtZXRyaWMgbsOgeSBuaMawIGJlbmNobWFyayBTY2lGYWN0IGhv4bq3YyBi4bqxbmcgY2jhu6luZyB24buBIMawdSB0aOG6vyBj4bunYSBLUFIuIiwKICAgICAgICBdCiAgICAgICAgcmVwb3J0WyJyZXNlYXJjaF9zdW1tYXJ5Il1bImludGVycHJldGF0aW9uX3ZpIl0gPSAoCiAgICAgICAgICAgICJOb3RlYm9vayBjaOG7iSBjaOG6oXkgZGlhZ25vc3RpY19vbmx5IGLhurFuZyBk4buvIGxp4buHdSBtaW5oIGjhu41hIHThu7EgdOG6oW8gdsOsIGtow7RuZyB04bqjaSDEkcaw4bujYyBkYXRhc2V0IGPDsyBuaMOjbi4gIgogICAgICAgICAgICAiQ8OhYyBtZXRyaWMga2jDtG5nIHBo4bqjaSBr4bq/dCBxdeG6oyBiZW5jaG1hcmsgU2NpRmFjdCB2w6Aga2jDtG5nIGNobyBwaMOpcCBr4bq/dCBsdeG6rW4gS1BSIHThu5F0IGjGoW4gQkdFLiIKICAgICAgICApCiAgICAgICAgcmV0dXJuCiAgICBvYnNlcnZhdGlvbnMgPSBbCiAgICAgICAgKAogICAgICAgICAgICBmIlRyw6puIHtyZXBvcnQuZ2V0KCdkYXRhc2V0Jywge30pLmdldCgnc2VsZWN0ZWRfcXVlcnlfY291bnQnLCAnTicpfSBxdWVyeSB2w6AgY8O5bmcgIgogICAgICAgICAgICBmIntyZXBvcnQuZ2V0KCdkYXRhc2V0Jywge30pLmdldCgnY2FuZGlkYXRlX3Bhc3NhZ2VfY291bnQnLCAnTicpfSBjYW5kaWRhdGUgcGFzc2FnZXMsICIKICAgICAgICAgICAgZiJSZWNhbGxAMTAgY+G7p2EgS1BSPXtrcHJbJ3JlY2FsbF9hdF8xMCddOi40Zn0sIEJHRT17YmFzZWxpbmVbJ3JlY2FsbF9hdF8xMCddOi40Zn0gIgogICAgICAgICAgICBmIihkZWx0YSBLUFLiiJJCR0U9e2RlbHRhcy5nZXQoJ3JlY2FsbF9hdF8xMCcsIDAuMCk6Ky40Zn0pLiIKICAgICAgICApLAogICAgICAgICgKICAgICAgICAgICAgZiJuRENHQDEwIGPhu6dhIEtQUj17a3ByWyduZGNnX2F0XzEwJ106LjRmfSwgQkdFPXtiYXNlbGluZVsnbmRjZ19hdF8xMCddOi40Zn07ICIKICAgICAgICAgICAgIsSRw6J5IGNo4buJIGzDoCBr4bq/dCBxdeG6oyB0csOqbiBzYW1wbGUgdsOgIHJlc3RyaWN0ZWQgY2FuZGlkYXRlIHBvb2wgbsOgeS4iCiAgICAgICAgKSwKICAgICAgICAiU2NpRmFjdCBxcmVscyBsw6AgcmVsZXZhbmNlIGp1ZGdtZW50czsgY2jDum5nIGtow7RuZyBjaOG7qW5nIG1pbmggcGFzc2FnZSBo4buXIHRy4bujIGhheSBtw6J1IHRodeG6q24gdOG7q25nIGNsYWltLiIsCiAgICBdCiAgICByZXBvcnRbInJlc2VhcmNoX3N1bW1hcnkiXVsib2JzZXJ2YXRpb25zIl0gPSBvYnNlcnZhdGlvbnMKICAgIHJlcG9ydFsicmVzZWFyY2hfc3VtbWFyeSJdWyJpbnRlcnByZXRhdGlvbl92aSJdID0gKAogICAgICAgIGYiVHJvbmcgbeG6q3UgbsOgeSwgUmVjYWxsQDEwIGPhu6dhIEtQUiBsw6Age2twclsncmVjYWxsX2F0XzEwJ106LjRmfSB2w6AgY+G7p2EgQkdFIGzDoCAiCiAgICAgICAgZiJ7YmFzZWxpbmVbJ3JlY2FsbF9hdF8xMCddOi40Zn07IG5EQ0dAMTAgbOG6p24gbMaw4bujdCBsw6Age2twclsnbmRjZ19hdF8xMCddOi40Zn0gdsOgICIKICAgICAgICBmIntiYXNlbGluZVsnbmRjZ19hdF8xMCddOi40Zn0uIMSQw6J5IGzDoCBr4bq/dCBxdeG6oyBzYW1wbGVkIHRyw6puIHJlc3RyaWN0ZWQgY2FuZGlkYXRlIHBvb2w7ICIKICAgICAgICAiY2jGsGEgxJHhu6cgY8ahIHPhu58gxJHhu4Mga+G6v3QgbHXhuq1uIEtQUiDGsHUgdmnhu4d0IG7Ds2kgY2h1bmcgaG/hurdjIHN1eSByYSBwYXNzYWdlIGjhu5cgdHLhu6MvbcOidSB0aHXhuqtuIGNsYWltLiIKICAgICkKICAgIGlmIHJlcG9ydC5nZXQoInRpbWluZ3MiLCB7fSkuZ2V0KCJxdWVyeV9lbmNvZGluZ19zZWNvbmRzX3Blcl9xdWVyeSIpOgogICAgICAgIHJlcG9ydFsicmVzZWFyY2hfc3VtbWFyeSJdWyJvYnNlcnZhdGlvbnMiXS5hcHBlbmQoCiAgICAgICAgICAgICJUaOG7nWkgZ2lhbiBtb2RlbCBsb2FkaW5nLCBxdWVyeSBlbmNvZGluZywgcGFzc2FnZSBlbmNvZGluZyB2w6Agc2NvcmluZyDEkcaw4bujYyBnaGkgcmnDqm5nOyAiCiAgICAgICAgICAgICJxdWVyeSBsYXRlbmN5IGtow7RuZyBn4buTbSBkb3dubG9hZCBoYXkgbW9kZWwgbG9hZGluZy4iCiAgICAgICAgKQoKCmRlZiBmaW5pc2hfZXhlY3V0aW9uKHJlcG9ydDogZGljdFtzdHIsIEFueV0sICosIGJvdGhfbW9kZWxzX2V2YWx1YXRlZDogYm9vbCkgLT4gTm9uZToKICAgIGhhc19lcnJvcnMgPSBib29sKHJlcG9ydFsiZXhlY3V0aW9uIl0uZ2V0KCJlcnJvcnMiKSkKICAgIGlmIGJvdGhfbW9kZWxzX2V2YWx1YXRlZCBhbmQgbm90IGhhc19lcnJvcnM6CiAgICAgICAgc3RhdHVzID0gInN1Y2Nlc3MiCiAgICBlbGlmIHJlcG9ydC5nZXQoImRhdGFzZXQiLCB7fSkuZ2V0KCJzZWxlY3RlZF9xdWVyeV9jb3VudCIpIG9yIGFueSgKICAgICAgICBpc2luc3RhbmNlKHZhbHVlLCBkaWN0KSBmb3IgdmFsdWUgaW4gcmVwb3J0LmdldCgibWV0cmljcyIsIHt9KS52YWx1ZXMoKQogICAgKToKICAgICAgICBzdGF0dXMgPSAicGFydGlhbF9zdWNjZXNzIgogICAgZWxzZToKICAgICAgICBzdGF0dXMgPSAiZmFpbGVkIgogICAgcmVwb3J0WyJleGVjdXRpb24iXVsic3RhdHVzIl0gPSBzdGF0dXMKCg=="}
for relative_path, encoded in EMBEDDED_SOURCE_B64.items():
    target = RUNTIME_ROOT / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(base64.b64decode(encoded))
sys.path.insert(0, str(RUNTIME_ROOT))
from src.reporting import build_empty_report, save_report, record_exception, collect_environment, collect_runtime_resources, update_research_summary, finish_execution

REPORT = build_empty_report()
RUN_STARTED = time.monotonic()
SOFT_DEADLINE, HARD_DEADLINE = RUN_STARTED + 25 * 60, RUN_STARTED + 30 * 60
REPORT["execution"]["hard_deadline_seconds"] = 1800
REPORT["execution"]["soft_deadline_seconds"] = 1500
REPORT["metadata"].update({"output_path": str(OUTPUT_PATH), "runtime_root": str(RUNTIME_ROOT)})
STOP_PIPELINE, DATA = False, None
MODEL_OBJECTS, EMBEDDINGS, MODEL_SNAPSHOTS = {}, {}, {}
HUB_MODEL_INFO, SNAPSHOT_PATTERNS, CACHED_SNAPSHOT_PATHS = {}, {}, {}

def both_models_evaluated():
    metrics = REPORT.get("metrics", {})
    return isinstance(metrics.get("baseline"), dict) and isinstance(metrics.get("kpr"), dict)

def save_progress():
    REPORT["timings"]["elapsed_seconds"] = time.monotonic() - RUN_STARTED
    if time.monotonic() >= SOFT_DEADLINE:
        REPORT["execution"]["soft_deadline_reached"] = True
    save_report(REPORT, OUTPUT_PATH)

def complete_step(name):
    if name not in REPORT["execution"]["completed_steps"]:
        REPORT["execution"]["completed_steps"].append(name)
    if both_models_evaluated() and not REPORT["execution"]["errors"]:
        REPORT["execution"]["status"] = "success"
    elif REPORT.get("dataset", {}).get("selected_query_count") or both_models_evaluated():
        REPORT["execution"]["status"] = "partial_success"
    save_progress()

def fail_stage(stage, error):
    global STOP_PIPELINE
    record_exception(REPORT, stage, error)
    REPORT["execution"]["status"] = "partial_success" if REPORT.get("dataset", {}).get("selected_query_count") or both_models_evaluated() else "failed"
    STOP_PIPELINE = True
    save_progress()
    print(f"Stage stopped: {stage}: {type(error).__name__}: {error}")

def check_soft_deadline(stage):
    if time.monotonic() >= SOFT_DEADLINE:
        REPORT["execution"]["soft_deadline_reached"] = True
        raise TimeoutError(f"25-minute soft deadline reached before/during {stage}.")

def disk_snapshot(path):
    import shutil
    usage = shutil.disk_usage(path)
    return {"path": str(path), "total_bytes": usage.total, "used_bytes": usage.used, "free_bytes": usage.free}

save_progress()
print(f"Project helpers ready. Report path: {OUTPUT_PATH}")


Project helpers ready. Report path: /kaggle/working/kpr_eval_report.json


## 1. Environment and GPU inspection

Capture Python, package versions, RAM, GPU, and cache disk before dependency changes. The existing Kaggle PyTorch/CUDA build is retained.

In [2]:
if not STOP_PIPELINE:
    try:
        REPORT["environment"] = collect_environment(include_torch=False)
        REPORT["environment"]["kaggle_runtime"] = IS_KAGGLE
        REPORT["environment"]["cache_disk_initial"] = disk_snapshot(CACHE_ROOT)
        REPORT["environment"]["report_disk_initial"] = disk_snapshot(OUTPUT_PATH.parent)
        import subprocess
        gpu_probe = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10)
        REPORT["environment"]["nvidia_smi_snapshot"] = gpu_probe.stdout.strip() if gpu_probe.returncode == 0 else None
        if gpu_probe.returncode != 0:
            REPORT["environment"]["nvidia_smi_probe_error"] = (gpu_probe.stderr or "nvidia-smi unavailable")[-1000:]
        DEVICE = "cpu"  # Replaced with the torch/CUDA probe after dependency installation.
        complete_step("environment_and_gpu_inspection")
        print(json.dumps({"nvidia_smi": REPORT["environment"]["nvidia_smi_snapshot"], "disk": REPORT["environment"]["cache_disk_initial"], "memory": REPORT["environment"].get("system_memory_bytes")}, indent=2))
    except Exception as exc:
        fail_stage("environment_inspection", exc)

{
  "nvidia_smi": "Tesla T4, 15360, 14912\nTesla T4, 15360, 14912",
  "disk": {
    "path": "/tmp/kpr_eval_cache",
    "total_bytes": 8758385086464,
    "used_bytes": 7720169971712,
    "free_bytes": 1038198337536
  },
  "memory": {
    "total": 33658155008,
    "available": 32172634112
  }
}


## 2. Dependency installation and validation

Install pinned inference dependencies without pinning or reinstalling `torch`.

In [3]:
if not STOP_PIPELINE:
    import subprocess, importlib.metadata
    started = time.monotonic()
    try:
        check_soft_deadline("dependency installation")
        packages = ["sentence-transformers==5.1.1", "transformers==4.55.4", "huggingface-hub==0.35.3", "safetensors==0.6.2", "numpy>=1.24,<3", "psutil>=5.9,<8", "pyarrow>=15,<24", "pandas>=2.0,<3", "spacy==3.8.18", "marisa-trie==1.4.1"]
        result = subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *packages], timeout=max(1, int(SOFT_DEADLINE - time.monotonic())))
        if result.returncode:
            raise RuntimeError(f"pip install exited with code {result.returncode}; see the pip output above.")
        import torch, sentence_transformers, transformers, huggingface_hub
        required = {"sentence-transformers": "5.1.1", "transformers": "4.55.4", "huggingface-hub": "0.35.3", "safetensors": "0.6.2", "spacy": "3.8.18", "marisa-trie": "1.4.1"}
        actual = {name: importlib.metadata.version(name) for name in required}
        mismatch = {name: {"expected": version, "actual": actual[name]} for name, version in required.items() if actual[name] != version}
        if mismatch:
            raise RuntimeError(f"Pinned dependency validation failed: {mismatch}")
        REPORT["timings"]["dependency_install_seconds"] = time.monotonic() - started
        REPORT["environment"]["package_versions"] = collect_environment(include_torch=True)["package_versions"]
        REPORT["environment"].update({"torch_cuda_build": torch.version.cuda, "cuda_available_after_install": bool(torch.cuda.is_available())})
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        REPORT["environment"]["selected_device"] = DEVICE
        if DEVICE == "cpu":
            REPORT["execution"]["warnings"].append("CUDA unavailable after dependency installation; using CPU inference.")
        complete_step("dependency_installation_and_validation")
        print(json.dumps({"pinned": actual, "torch": torch.__version__, "torch_cuda": torch.version.cuda, "device": DEVICE}, indent=2))
    except Exception as exc:
        REPORT["timings"]["dependency_install_seconds"] = time.monotonic() - started
        fail_stage("dependency_installation", exc)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 30.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 15.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 68.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 74.9 MB/s  0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.8.0
    Uninstalling safetensors-0.8.0:
      Successfully uninstalled safetensors-0.8.0
  Attempting uninstall: confection
    Found existing installation: confection 1.3.3
    Uninstalling confection-1.3.3:
      Successfully uninstalled confection-1.3.3
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    U

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.35.3 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.35.3 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.35.3 which is incompatible.
diffusers 0.42.0 requires safetensors>=0.8.0, but you have safetensors 0.6.2 which is incompatible.
/usr/local/lib/python3.13/dist-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
I0000 00:00:1794929858.322367      50 cudart_stub.cc:31] Could not find cuda d

{
  "pinned": {
    "sentence-transformers": "5.1.1",
    "transformers": "4.55.4",
    "huggingface-hub": "0.35.3",
    "safetensors": "0.6.2",
    "spacy": "3.8.18",
    "marisa-trie": "1.4.1"
  },
  "torch": "2.11.0+cu130",
  "torch_cuda": "13.0",
  "device": "cuda"
}


## 3. Official repository, model-card, and checkpoint inspection

Clone and inspect official KPR inference code. Read model files and size metadata at the pinned Hub revision before any full checkpoint download.

In [4]:
if not STOP_PIPELINE:
    import subprocess, uuid
    started = time.monotonic()
    try:
        check_soft_deadline("official source inspection")
        from src.models import KPR_MODEL_ID, BASELINE_MODEL_ID, inspect_hub_model, inspect_kpr_remote_code, required_snapshot_files
        repo = CACHE_ROOT / f"official_repository_{uuid.uuid4().hex}"
        clone = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/knowledgeable-embedding/knowledgeable-embedding.git", str(repo)], capture_output=True, text=True, timeout=max(1, min(180, int(SOFT_DEADLINE - time.monotonic()))))
        if clone.returncode:
            raise RuntimeError(f"Official KPR clone failed: {(clone.stderr or clone.stdout)[-3000:]}")
        commit = subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True, text=True, check=True, timeout=30).stdout.strip()
        readme = (repo / "README.md").read_text(encoding="utf-8")
        source = (repo / "src/kembed/modeling.py").read_text(encoding="utf-8")
        repo_checks = {"sentence_transformer_api": "SentenceTransformer" in readme, "trust_remote_code_api": "trust_remote_code=True" in readme, "entity_attention": "EntityFusionMultiHeadAttention" in source, "entity_fusion": "EntityFusionLayer" in source, "entity_inputs": "entity_embeds" in source and "entity_position_ids" in source}
        if not all(repo_checks.values()):
            raise RuntimeError(f"Official inference-source checks failed: {repo_checks}")
        kpr_info, baseline_info = inspect_hub_model(KPR_MODEL_ID), inspect_hub_model(BASELINE_MODEL_ID)
        SNAPSHOT_PATTERNS["kpr"], kpr_bytes = required_snapshot_files(kpr_info)
        SNAPSHOT_PATTERNS["baseline"], baseline_bytes = required_snapshot_files(baseline_info)
        code_check = inspect_kpr_remote_code(KPR_MODEL_ID, kpr_info["revision"], HF_CACHE)
        from huggingface_hub import hf_hub_download, snapshot_download
        card_path = hf_hub_download(repo_id=KPR_MODEL_ID, filename="README.md", revision=kpr_info["revision"], cache_dir=str(HF_CACHE), etag_timeout=30)
        card = Path(card_path).read_text(encoding="utf-8")
        card_checks = {"sentence_transformer_api": "SentenceTransformer" in card, "trust_remote_code_api": "trust_remote_code=True" in card}
        if not all(card_checks.values()):
            raise RuntimeError(f"KPR model-card checks failed: {card_checks}")
        def cached_snapshot(info, patterns):
            try:
                cached_path = snapshot_download(repo_id=info["model_id"], revision=info["revision"], cache_dir=str(HF_CACHE), allow_patterns=patterns, local_files_only=True)
                return cached_path if all((Path(cached_path) / name).is_file() for name in patterns) else None
            except Exception:
                return None
        CACHED_SNAPSHOT_PATHS["kpr"] = cached_snapshot(kpr_info, SNAPSHOT_PATTERNS["kpr"])
        CACHED_SNAPSHOT_PATHS["baseline"] = cached_snapshot(baseline_info, SNAPSHOT_PATTERNS["baseline"])
        HUB_MODEL_INFO.update({"kpr": kpr_info, "baseline": baseline_info})
        REPORT["model_information"]["kpr"] = {"model_id": KPR_MODEL_ID, "revision": kpr_info["revision"], "commit_url": kpr_info["commit_url"], "hub_repository_size_bytes": kpr_info["total_repository_size_bytes"], "required_inference_checkpoint_bytes": kpr_bytes, "required_inference_files": SNAPSHOT_PATTERNS["kpr"], "complete_checkpoint_cache_hit": CACHED_SNAPSHOT_PATHS["kpr"] is not None, "remote_code_inspection": code_check, "model_card_checks": card_checks}
        REPORT["model_information"]["baseline"] = {"model_id": BASELINE_MODEL_ID, "revision": baseline_info["revision"], "commit_url": baseline_info["commit_url"], "hub_repository_size_bytes": baseline_info["total_repository_size_bytes"], "required_inference_checkpoint_bytes": baseline_bytes, "required_inference_files": SNAPSHOT_PATTERNS["baseline"], "complete_checkpoint_cache_hit": CACHED_SNAPSHOT_PATHS["baseline"] is not None}
        REPORT["metadata"]["official_implementation_inspection"] = {"repository_url": "https://github.com/knowledgeable-embedding/knowledgeable-embedding", "repository_commit": commit, "repository_checks": repo_checks, "checkpoint_source_revision": kpr_info["revision"], "custom_source_checks": code_check["checks"], "model_card_checks": card_checks, "custom_code_pinned_to_inspected_revision": True}
        REPORT["timings"]["repository_inspection_seconds"] = time.monotonic() - started
        complete_step("official_repository_and_checkpoint_inspection")
        print(json.dumps({"repository_commit": commit, "kpr_revision": kpr_info["revision"], "kpr_required_bytes": kpr_bytes, "baseline_required_bytes": baseline_bytes, "custom_code_checks": code_check["checks"]}, indent=2))
    except Exception as exc:
        REPORT["timings"]["repository_inspection_seconds"] = time.monotonic() - started
        fail_stage("official_source_and_checkpoint_inspection", exc)

configuration.py: 0.00B [00:00, ?B/s]

modeling.py: 0.00B [00:00, ?B/s]

tokenization_kpr.py: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

{
  "repository_commit": "fd163ed8ae8f02f6f85581dc62efe040ac475c11",
  "kpr_revision": "2cca6d26e56a02390b07247fc567afe34a46a184",
  "kpr_required_bytes": 12195308648,
  "baseline_required_bytes": 1312709295,
  "custom_code_checks": {
    "entity_fusion_attention_present": true,
    "entity_fusion_layer_present": true,
    "entity_inputs_consumed": true,
    "memory_mapped_entity_embeddings": true,
    "custom_bert_configuration_present": true,
    "entity_linking_and_positions_present": true,
    "marisa_trie_dependency_present": true,
    "spacy_dependency_present": true
  }
}


## 4. SciFact download, fallback, and deterministic sampling

Use the official BEIR archive first, then the BEIR Hub mirror and matching qrels. If both fail, label the fallback fixture `diagnostic_only`.

In [5]:
if not STOP_PIPELINE:
    from src.data import download_scifact_archive, load_and_sample_scifact, load_and_sample_scifact_hf_mirror, diagnostic_demo_sample
    started, prep_seconds, attempts = time.monotonic(), 0.0, []
    try:
        check_soft_deadline("SciFact data acquisition")
        try:
            archive = download_scifact_archive(DATA_CACHE, deadline_monotonic=SOFT_DEADLINE)
            prep = time.monotonic()
            DATA = load_and_sample_scifact(archive["path"], target_queries=40, target_candidates=800, random_seed=42)
            prep_seconds = time.monotonic() - prep
            DATA["metadata"].update({"archive_bytes": archive["bytes"], "archive_cache_hit": archive["cache_hit"]})
        except Exception as primary_error:
            attempts.append({"source": "official_beir_archive", "status": "failed", "error_type": type(primary_error).__name__, "message": str(primary_error)})
            print(f"Primary SciFact source failed; trying BEIR Hub mirror: {primary_error}")
            check_soft_deadline("SciFact Hub mirror fallback")
            try:
                mirror_started = time.monotonic()
                DATA = load_and_sample_scifact_hf_mirror(DATA_CACHE, target_queries=40, target_candidates=800, random_seed=42)
                prep_seconds += DATA["metadata"].get("preparation_seconds", 0.0)
                attempts.append({"source": "BeIR/scifact plus BeIR/scifact-qrels", "status": "success"})
            except Exception as mirror_error:
                attempts.append({"source": "official_beir_huggingface_mirror", "status": "failed", "error_type": type(mirror_error).__name__, "message": str(mirror_error)})
                DATA = diagnostic_demo_sample()
                attempts.append({"source": "hand_authored_fixture", "status": "success", "evaluation_type": "diagnostic_only"})
        REPORT["timings"]["dataset_download_seconds"] = time.monotonic() - started - prep_seconds
        REPORT["timings"]["dataset_preparation_seconds"] = prep_seconds
        metadata = dict(DATA["metadata"])
        metadata.update({"selected_query_ids": DATA["query_ids"], "selected_query_count": len(DATA["query_ids"]), "candidate_passage_count": len(DATA["candidate_ids"]), "positive_qrel_count": sum(map(len, DATA["qrels"].values())), "fallback_attempts": attempts})
        REPORT["dataset"] = metadata
        evaluation_type = metadata.get("evaluation_type", "restricted_candidate_retrieval")
        REPORT["evaluation_protocol"]["evaluation_type"] = evaluation_type
        REPORT["evaluation_protocol"]["dataset_fallback_attempts"] = attempts
        if evaluation_type == "diagnostic_only":
            REPORT["execution"]["warnings"].append("No labeled SciFact source loaded. Any metrics are diagnostic_only, not a benchmark.")
            REPORT["evaluation_protocol"]["qrel_interpretation"] = "Hand-authored diagnostic labels are only a workflow fixture, not SciFact relevance or factual support/contradiction judgments."
            REPORT["evaluation_protocol"]["candidate_pool_note"] = "Hand-authored diagnostic fixture only; not a real benchmark."
        elif len(DATA["query_ids"]) < 30:
            REPORT["execution"]["warnings"].append(f"Only {len(DATA['query_ids'])} labeled queries were available; below the 30-query target.")
        complete_step("dataset_download_and_sampling")
        print(json.dumps({"dataset": metadata.get("name"), "evaluation_type": evaluation_type, "queries": len(DATA["query_ids"]), "candidates": len(DATA["candidate_ids"]), "fallbacks": attempts}, indent=2))
    except Exception as exc:
        REPORT["timings"]["dataset_download_seconds"] = time.monotonic() - started
        REPORT["timings"]["dataset_preparation_seconds"] = prep_seconds
        fail_stage("dataset_download_and_sampling", exc)

{
  "dataset": "BEIR SciFact",
  "evaluation_type": "restricted_candidate_retrieval",
  "queries": 40,
  "candidates": 800,
  "fallbacks": []
}


## 5. KPR size preflight, checkpoint download, and loading

KPR is loaded only from the inspected, pinned Hub revision. The large entity table is required by its tokenizer and is memory-mapped on CPU, not copied to GPU.

In [6]:
if not STOP_PIPELINE:
    import shutil
    from src.models import download_snapshot_with_deadline, cache_size_bytes, load_sentence_transformer
    try:
        check_soft_deadline("KPR checkpoint preflight")
        km, bm = REPORT["model_information"]["kpr"], REPORT["model_information"]["baseline"]
        need_k = 0 if CACHED_SNAPSHOT_PATHS["kpr"] else km["required_inference_checkpoint_bytes"]
        need_b = 0 if CACHED_SNAPSHOT_PATHS["baseline"] else bm["required_inference_checkpoint_bytes"]
        reserve = 1_500_000_000
        free = shutil.disk_usage(CACHE_ROOT).free
        preflight = {"cache_path": str(HF_CACHE), "free_bytes_before_model_download": free, "required_kpr_bytes_if_uncached": need_k, "required_baseline_bytes_if_uncached": need_b, "reserved_headroom_bytes": reserve, "required_free_bytes": need_k + need_b + reserve, "preflight_passed": free >= need_k + need_b + reserve}
        REPORT["resource_usage"].update({"checkpoint_disk_preflight": preflight, "disk_before_model_download": disk_snapshot(CACHE_ROOT)})
        if not preflight["preflight_passed"]:
            raise OSError(f"Insufficient disk: need about {preflight['required_free_bytes']:,} free bytes; found {free:,}. KPR was not downloaded or substituted.")
        save_progress()
        check_soft_deadline("KPR checkpoint download")
        if CACHED_SNAPSHOT_PATHS["kpr"]:
            snapshot, cache_delta = CACHED_SNAPSHOT_PATHS["kpr"], 0
            REPORT["timings"]["model_download_seconds"]["kpr"] = 0.0
        else:
            before, began = cache_size_bytes(HF_CACHE), time.monotonic()
            try:
                snapshot = download_snapshot_with_deadline(km["model_id"], km["revision"], HF_CACHE, SNAPSHOT_PATTERNS["kpr"], stop_at_monotonic=SOFT_DEADLINE, max_workers=4)
            finally:
                REPORT["timings"]["model_download_seconds"]["kpr"] = time.monotonic() - began
            cache_delta = max(0, cache_size_bytes(HF_CACHE) - before)
        MODEL_SNAPSHOTS["kpr"] = snapshot
        REPORT["model_information"]["kpr"].update({"download_completed": True, "actual_cache_bytes_added_estimate": cache_delta})
        save_progress()
        load_started = time.monotonic()
        try:
            model, loaded = load_sentence_transformer(snapshot, device=DEVICE, trust_remote_code=True, max_seq_length=256)
        except Exception:
            REPORT["timings"]["model_load_seconds"]["kpr"] = time.monotonic() - load_started
            raise
        MODEL_OBJECTS["kpr"] = model
        loaded.update({"actual_cache_bytes_added_estimate": cache_delta, "required_checkpoint_download_bytes": km["required_inference_checkpoint_bytes"], "hub_repository_size_bytes": km["hub_repository_size_bytes"], "model_load_memory_usage": collect_runtime_resources(loaded["device"])})
        REPORT["timings"]["model_load_seconds"]["kpr"] = loaded["load_seconds"]
        REPORT["model_information"]["kpr"].update(loaded)
        if loaded["load_fallback"]:
            REPORT["execution"]["warnings"].append(loaded["load_fallback"])
        REPORT["resource_usage"]["disk_after_kpr_download"] = disk_snapshot(CACHE_ROOT)
        complete_step("kpr_checkpoint_download_and_loading")
        print(json.dumps({"revision": km["revision"], "load_seconds": loaded["load_seconds"], "dimension": loaded["embedding_dimension"], "device": loaded["device"], "precision": loaded["precision"], "disk": REPORT["resource_usage"]["disk_after_kpr_download"]}, indent=2))
    except Exception as exc:
        fail_stage("kpr_checkpoint_preflight_download_or_load", exc)

Fetching 19 files: 100%|██████████| 19/19 [00:46<00:00,  2.47s/it]


{
  "revision": "2cca6d26e56a02390b07247fc567afe34a46a184",
  "load_seconds": 19.384546537999995,
  "dimension": 768,
  "device": "cuda:0",
  "precision": "float32",
  "disk": {
    "path": "/tmp/kpr_eval_cache",
    "total_bytes": 8758385086464,
    "used_bytes": 7732826329088,
    "free_bytes": 1025541980160
  }
}


## 6. Baseline BGE checkpoint download and loading

Fetch the pinned base model from the same Hugging Face cache and report download and loading times separately.

In [7]:
if not STOP_PIPELINE:
    try:
        check_soft_deadline("baseline checkpoint download")
        from src.models import download_snapshot_with_deadline, cache_size_bytes, load_sentence_transformer
        bm = REPORT["model_information"]["baseline"]
        if CACHED_SNAPSHOT_PATHS["baseline"]:
            snapshot, cache_delta = CACHED_SNAPSHOT_PATHS["baseline"], 0
            REPORT["timings"]["model_download_seconds"]["baseline"] = 0.0
        else:
            before, began = cache_size_bytes(HF_CACHE), time.monotonic()
            try:
                snapshot = download_snapshot_with_deadline(bm["model_id"], bm["revision"], HF_CACHE, SNAPSHOT_PATTERNS["baseline"], stop_at_monotonic=SOFT_DEADLINE, max_workers=4)
            finally:
                REPORT["timings"]["model_download_seconds"]["baseline"] = time.monotonic() - began
            cache_delta = max(0, cache_size_bytes(HF_CACHE) - before)
        MODEL_SNAPSHOTS["baseline"] = snapshot
        REPORT["model_information"]["baseline"].update({"download_completed": True, "actual_cache_bytes_added_estimate": cache_delta})
        save_progress()
        load_started = time.monotonic()
        try:
            model, loaded = load_sentence_transformer(snapshot, device=DEVICE, trust_remote_code=False, max_seq_length=256)
        except Exception:
            REPORT["timings"]["model_load_seconds"]["baseline"] = time.monotonic() - load_started
            raise
        MODEL_OBJECTS["baseline"] = model
        loaded.update({"actual_cache_bytes_added_estimate": cache_delta, "required_checkpoint_download_bytes": bm["required_inference_checkpoint_bytes"], "hub_repository_size_bytes": bm["hub_repository_size_bytes"], "model_load_memory_usage": collect_runtime_resources(loaded["device"])})
        REPORT["timings"]["model_load_seconds"]["baseline"] = loaded["load_seconds"]
        REPORT["model_information"]["baseline"].update(loaded)
        if loaded["load_fallback"]:
            REPORT["execution"]["warnings"].append(loaded["load_fallback"])
        REPORT["resource_usage"]["disk_after_baseline_download"] = disk_snapshot(CACHE_ROOT)
        if loaded["embedding_dimension"] != REPORT["model_information"]["kpr"].get("embedding_dimension"):
            raise ValueError(f"Embedding dimensions differ: KPR={REPORT['model_information']['kpr'].get('embedding_dimension')}, BGE={loaded['embedding_dimension']}.")
        complete_step("baseline_checkpoint_download_and_loading")
        print(json.dumps({"revision": bm["revision"], "load_seconds": loaded["load_seconds"], "dimension": loaded["embedding_dimension"], "device": loaded["device"], "precision": loaded["precision"], "disk": REPORT["resource_usage"]["disk_after_baseline_download"]}, indent=2))
    except Exception as exc:
        fail_stage("baseline_checkpoint_download_or_load", exc)

Fetching 12 files: 100%|██████████| 12/12 [00:06<00:00,  1.90it/s]


{
  "revision": "a5beb1e3e68b9ab74eb54cfd186867f64f240e1a",
  "load_seconds": 0.25913628799997923,
  "dimension": 768,
  "device": "cuda:0",
  "precision": "float32",
  "disk": {
    "path": "/tmp/kpr_eval_cache",
    "total_bytes": 8758385086464,
    "used_bytes": 7734139215872,
    "free_bytes": 1024229093376
  }
}


## 7–8. Warm-up, query/passage embeddings, and inference timing

Both models use the same sampled queries and candidates. BGE's recommended retrieval instruction is added to BGE queries only; KPR uses raw text per its SentenceTransformer model-card example. Query latency excludes download, model loading, and warm-up.

In [8]:
if not STOP_PIPELINE:
    from src.data import passage_text
    from src.models import encode_texts_batched
    try:
        check_soft_deadline("embedding generation")
        query_ids, candidate_ids = DATA["query_ids"], DATA["candidate_ids"]
        raw_queries = [DATA["queries"][qid] for qid in query_ids]
        passages = [passage_text(DATA["corpus"][did]) for did in candidate_ids]
        instruction = "Represent this sentence for searching relevant passages: "
        model_queries = {"kpr": raw_queries, "baseline": [instruction + query for query in raw_queries]}
        for key in ("kpr", "baseline"):
            check_soft_deadline(f"{key} warm-up")
            model = MODEL_OBJECTS[key]
            warmup_started = time.monotonic()
            _, warmup_stats = encode_texts_batched(model, [model_queries[key][0]], batch_size=1, deadline_monotonic=SOFT_DEADLINE, description=f"{key} warm-up")
            REPORT["model_information"][key]["warmup"] = warmup_stats
            if warmup_stats["cpu_fallbacks"]:
                REPORT["execution"]["warnings"].append(f"{key} switched to CPU during warm-up after CUDA out-of-memory.")
            REPORT["timings"]["warmup_seconds"][key] = time.monotonic() - warmup_started
            qemb, qstats = encode_texts_batched(model, model_queries[key], batch_size=8, deadline_monotonic=SOFT_DEADLINE, description=f"{key} queries")
            EMBEDDINGS[f"{key}_queries"] = qemb
            REPORT["timings"]["query_encoding_seconds"][key] = qstats["seconds"]
            REPORT["timings"]["query_encoding_seconds_per_query"][key] = qstats["seconds_per_item"]
            REPORT["model_information"][key]["query_encoding"] = qstats
            save_progress()
            pemb, pstats = encode_texts_batched(model, passages, batch_size=8, deadline_monotonic=SOFT_DEADLINE, description=f"{key} passages")
            EMBEDDINGS[f"{key}_passages"] = pemb
            REPORT["timings"]["passage_encoding_seconds"][key] = pstats["seconds"]
            REPORT["model_information"][key]["passage_encoding"] = pstats
            REPORT["model_information"][key]["effective_inference_device"] = pstats["effective_device"]
            if qstats["cpu_fallbacks"] + pstats["cpu_fallbacks"]:
                REPORT["execution"]["warnings"].append(f"{key} switched to CPU after CUDA out-of-memory; check memory and timings in the report.")
            REPORT["model_information"][key]["inference_memory_usage"] = collect_runtime_resources(pstats["effective_device"])
            if qemb.shape[1] != pemb.shape[1]:
                raise ValueError(f"{key} query and passage dimensions differ.")
            save_progress()
            print(f"{key} queries={qemb.shape}, passages={pemb.shape}, batch={pstats['effective_batch_size']}")
        complete_step("query_and_passage_embedding_generation")
    except Exception as exc:
        fail_stage("embedding_generation", exc)

/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2751: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


kpr queries=(40, 768), passages=(800, 768), batch=8
baseline queries=(40, 768), passages=(800, 768), batch=8


## 9–11. Similarity scoring, retrieval metrics, and paired failure analysis

Compute metrics only over the shared restricted candidate pool. Preserve query, full top-10 passage text, score, rank, qrel, and both models' result. Failure categories remain unclassified unless supported by labels.

In [9]:
if not STOP_PIPELINE:
    from src.evaluation import evaluate_retrieval, combine_query_results, metric_deltas
    try:
        check_soft_deadline("retrieval evaluation")
        evaluation_started = time.monotonic()
        base = evaluate_retrieval(EMBEDDINGS["baseline_queries"], EMBEDDINGS["baseline_passages"], DATA["query_ids"], DATA["candidate_ids"], DATA["qrels"], DATA["corpus"])
        kpr = evaluate_retrieval(EMBEDDINGS["kpr_queries"], EMBEDDINGS["kpr_passages"], DATA["query_ids"], DATA["candidate_ids"], DATA["qrels"], DATA["corpus"])
        REPORT["timings"]["scoring_seconds"].update({"baseline": base["scoring_seconds"], "kpr": kpr["scoring_seconds"]})
        REPORT["metrics"]["baseline"], REPORT["metrics"]["kpr"] = base["metrics"], kpr["metrics"]
        REPORT["metrics"]["comparison"] = {"delta_kpr_minus_baseline": metric_deltas(base["metrics"], kpr["metrics"]), "interpretation_scope": "paired sampled restricted-candidate experiment only"}
        per_query, failures = combine_query_results(DATA["query_ids"], DATA["queries"], DATA["qrels"], base, kpr)
        REPORT["per_query_results"], REPORT["failure_analysis"] = per_query, failures
        REPORT["timings"]["evaluation_seconds"] = time.monotonic() - evaluation_started
        REPORT["model_information"]["baseline"]["scored_pairs"] = base["scored_pairs"]
        REPORT["model_information"]["kpr"]["scored_pairs"] = kpr["scored_pairs"]
        REPORT["resource_usage"].update({"final_runtime_snapshot": collect_runtime_resources(DEVICE), "disk_after_evaluation": disk_snapshot(CACHE_ROOT)})
        update_research_summary(REPORT)
        complete_step("similarity_scoring_metrics_comparison_and_failure_analysis")
        print(json.dumps({"baseline": base["metrics"], "kpr": kpr["metrics"], "delta_kpr_minus_baseline": REPORT["metrics"]["comparison"]["delta_kpr_minus_baseline"], "failure_case_counts": failures["counts"]}, indent=2))
    except Exception as exc:
        REPORT["timings"]["evaluation_seconds"] = time.monotonic() - evaluation_started if "evaluation_started" in globals() else None
        fail_stage("retrieval_evaluation_and_failure_analysis", exc)

{
  "baseline": {
    "recall_at_1": 0.6958333333333334,
    "recall_at_5": 0.875,
    "recall_at_10": 0.9,
    "mrr_at_10": 0.8069444444444442,
    "ndcg_at_10": 0.8311838264322591
  },
  "kpr": {
    "recall_at_1": 0.6708333333333334,
    "recall_at_5": 0.825,
    "recall_at_10": 0.9,
    "mrr_at_10": 0.7777777777777778,
    "ndcg_at_10": 0.8078353394866857
  },
  "delta_kpr_minus_baseline": {
    "recall_at_1": -0.025000000000000022,
    "recall_at_5": -0.050000000000000044,
    "recall_at_10": 0.0,
    "mrr_at_10": -0.029166666666666452,
    "ndcg_at_10": -0.023348486945573343
  },
  "failure_case_counts": {
    "kpr_better_available": 3,
    "baseline_better_available": 6,
    "both_failed_available": 3,
    "examples_returned_per_category": 5,
    "selection_metric": "per-query nDCG@10; both_failed means zero Recall@10 for both models"
  }
}


## 12. Consolidated JSON export

Refresh and parse-verify the one result file. Uncomputed metrics remain `null`; download, disk, memory, deadline, and code-loading blockers stay in the report.

In [10]:
both_done = both_models_evaluated()
if both_done:
    update_research_summary(REPORT)
finish_execution(REPORT, both_models_evaluated=both_done)
REPORT["timings"]["elapsed_seconds"] = time.monotonic() - RUN_STARTED
if REPORT["timings"]["elapsed_seconds"] >= 1800:
    REPORT["execution"]["hard_deadline_exceeded"] = True
    REPORT["execution"]["warnings"].append("The 30-minute hard target was exceeded; runtime is not guaranteed across Kaggle environments.")
REPORT["resource_usage"]["final_disk"] = disk_snapshot(CACHE_ROOT)
REPORT["resource_usage"].setdefault("final_runtime_snapshot", collect_runtime_resources(REPORT.get("environment", {}).get("selected_device")))
export_started = time.monotonic()
report_path = save_report(REPORT, OUTPUT_PATH)
REPORT["timings"]["report_export_seconds"] = time.monotonic() - export_started
REPORT["timings"]["elapsed_seconds"] = time.monotonic() - RUN_STARTED
report_path = save_report(REPORT, OUTPUT_PATH)
with report_path.open("r", encoding="utf-8") as stream:
    verified_report = json.load(stream)
print(f"Report: {report_path}")
print(f"Execution status: {verified_report['execution']['status']}")
if isinstance(verified_report.get("metrics", {}).get("baseline"), dict) and isinstance(verified_report.get("metrics", {}).get("kpr"), dict):
    print("Recall@10 | BGE: {:.4f} | KPR: {:.4f}".format(verified_report["metrics"]["baseline"]["recall_at_10"], verified_report["metrics"]["kpr"]["recall_at_10"]))
    print("nDCG@10   | BGE: {:.4f} | KPR: {:.4f}".format(verified_report["metrics"]["baseline"]["ndcg_at_10"], verified_report["metrics"]["kpr"]["ndcg_at_10"]))
else:
    print("No paired metrics were computed; inspect execution.errors and resource_usage in the JSON.")
if verified_report.get("dataset", {}).get("evaluation_type") == "diagnostic_only":
    print("Diagnostic-only run: these values are not benchmark metrics.")

Report: /kaggle/working/kpr_eval_report.json
Execution status: success
Recall@10 | BGE: 0.9000 | KPR: 0.9000
nDCG@10   | BGE: 0.8312 | KPR: 0.8078
